# SC-GWO Brain Tumour Framework — YOLO baselines on BraTS 2019 and BraTS 2020

**How to run:** Add Input → *BraTS 2019* (aryashah2k/brain-tumor-segmentation-brats-2019) and *BraTS2020 Dataset (Training + Validation)* (awsaf49/brats20-dataset-training-validation). Turn Internet **ON** and GPU **ON**. Then **Save Version → Save & Run All (Commit)** and download `YOLO_RESULTS_both.zip` from the version's *Output* tab. The last cell also prints a text block to copy.

## 0. Setup — install dependencies

In [ ]:

# Kaggle notebooks already have torch/torchvision/numpy/scipy/scikit-learn/
# scikit-image/nibabel pre-installed. PyWavelets is the only common gap.
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("PyWavelets", "pywt"), ("nibabel", "nibabel")]:
    _ensure(pkg, imp)

print("Dependencies OK.")


## 0.1 Setup — downloading the data (skip this cell on Kaggle if you
used *Add Data* instead)

If you are running this notebook **outside** Kaggle (e.g. a local Jupyter
server or Colab), uncomment and run the cell below once, with a valid
Kaggle API token placed at `~/.kaggle/kaggle.json`.

In [ ]:

# import subprocess
# subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
# subprocess.run(["kaggle", "datasets", "download", "-d", "awsaf49/brats2020-training-data"], check=True)
# subprocess.run(["unzip", "-oq", "brats2020-training-data.zip", "-d", "./brats2020_raw"], check=True)
#
# subprocess.run(["kaggle", "datasets", "download", "-d", "aryashah2k/brain-tumor-segmentation-brats-2019"], check=True)
# subprocess.run(["unzip", "-oq", "brain-tumor-segmentation-brats-2019.zip", "-d", "./brats2019_raw"], check=True)
print("If running outside Kaggle: uncomment the lines above and re-run this cell.")


## 0.2 Auto-detect the dataset path

In [ ]:

import os, glob, re

def find_brats_root():
    # Searches common Kaggle/local locations for a BraTS-style folder tree
    # and returns the first plausible match, checking for BOTH known
    # formats: (a) per-patient/HGG-LGG NIfTI folders, and (b) the
    # per-slice .h5 format used by some Kaggle re-uploads (e.g. the
    # awsaf49/brats2020-training-data dataset).
    search_dirs = []
    if os.path.isdir("/kaggle/input"):
        search_dirs += glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
    search_dirs += glob.glob("./*raw*") + glob.glob("./*brats*") + glob.glob("./*BraTS*")

    def has_h5(d):
        for r, _, files in os.walk(d):
            if any(re.match(r"volume_\d+_slice_\d+\.h5$", f) for f in files):
                return True
        return False

    def has_nifti(d):
        if os.path.isdir(os.path.join(d, "HGG")) or os.path.isdir(os.path.join(d, "LGG")):
            return True
        return bool(glob.glob(os.path.join(d, "*", "*_seg.nii*")))

    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        if has_h5(d) or has_nifti(d):
            return d
        for sub in glob.glob(os.path.join(d, "*")):
            if os.path.isdir(sub) and (has_h5(sub) or has_nifti(sub)):
                return sub
    return None

AUTO_DATA_ROOT = find_brats_root()
if AUTO_DATA_ROOT:
    print(f"Auto-detected BraTS data root: {AUTO_DATA_ROOT}")
else:
    print("Could not auto-detect a BraTS folder. Set data_root manually in the Configuration cell.")
    if os.path.isdir("/kaggle/input"):
        print("Contents of /kaggle/input:", os.listdir("/kaggle/input"))


### 0.2.1 (Optional but recommended) Diagnostic: inspect what was found

Run this once to confirm the detected format before training. It's harmless
to run even if you're confident everything is correct.

In [ ]:

import re as _re

def diagnose(root):
    print(f"Inspecting: {root}\n")
    h5_files, nifti_seg_files = [], []
    for r, _, files in os.walk(root):
        h5_files += [os.path.join(r, f) for f in files if _re.match(r"volume_\d+_slice_\d+\.h5$", f)]
        nifti_seg_files += [os.path.join(r, f) for f in files if f.endswith("_seg.nii.gz") or f.endswith("_seg.nii")]

    if h5_files:
        print(f"Format detected: H5-per-slice ({len(h5_files)} files found)")
        print("Example file:", h5_files[0])
        try:
            import h5py
            with h5py.File(h5_files[0], "r") as f:
                print("Keys inside:", list(f.keys()))
                for k in f.keys():
                    print(f"  {k}: shape={f[k].shape}, dtype={f[k].dtype}")
        except ImportError:
            print("(install h5py to inspect file contents: !pip install -q h5py)")
        print("\nNOTE: this format does not include HGG/LGG grade labels.")
        print("The classification stage will be automatically skipped;")
        print("segmentation will run normally. See the markdown note below.")
    elif nifti_seg_files:
        print(f"Format detected: raw NIfTI ({len(nifti_seg_files)} segmentation volumes found)")
        print("Example file:", nifti_seg_files[0])
        name_mapping = glob.glob(os.path.join(root, "name_mapping.csv"))
        if name_mapping:
            print("Found name_mapping.csv -> grade labels should be available.")
        elif os.path.isdir(os.path.join(root, "HGG")) or os.path.isdir(os.path.join(root, "LGG")):
            print("Found HGG/LGG folder split -> grade labels available.")
        else:
            print("No grade-label source found -> classification stage will be skipped.")
    else:
        print("No recognised BraTS files found under this path. Check the path is correct.")

if AUTO_DATA_ROOT:
    diagnose(AUTO_DATA_ROOT)
else:
    print("Set data_root manually below, then re-run this cell with diagnose(data_root).")


> **About grade (benign/malignant) labels.** Some Kaggle re-uploads of
> BraTS2020 (including the popular `awsaf49/brats2020-training-data`) store
> data as pre-sliced `.h5` files and do **not** include HGG/LGG grade
> metadata. In that case, this notebook will:
> - run **segmentation** (Swin-ResSegNet, SC-GWO weight refinement) normally
>   and report Tables 12-13 metrics — this part is unaffected.
> - **automatically skip** the classification stage (WFA-ResNet, SC-GWO
>   feature selection) and print a warning saying so — you'll get `nan`/empty
>   values for Tables 14-15 from *this* dataset.
>
> If you need classification results too, use a BraTS download that
> includes grade labels — e.g. the raw NIfTI convention with a
> `name_mapping.csv` (Grade column) or an `HGG`/`LGG` folder split, which
> some other Kaggle mirrors and the BraTS 2019 `aryashah2k` dataset provide.
> The diagnostic cell above tells you which case you're in.

## 0.3 Configuration

Edit these values, then Run All.

In [ ]:

from types import SimpleNamespace
import torch

CONFIG = dict(
    data_root       = AUTO_DATA_ROOT or "/kaggle/input/brats2020-training-data/MICCAI_BraTS2020_TrainingData",
    dataset_name    = "BraTS2020",
    out_dir         = "./results",
    device          = "cuda" if torch.cuda.is_available() else "cpu",
    seed            = 0,
    min_tumour_pixels = 50,

    # Table 7 (Section 4.4): gradient-based training
    epochs_seg      = 100,
    lr_seg          = 1e-4,
    patience_seg    = 15,
    batch_size_seg  = 16,

    # Table 8 (Section 4.4): SC-GWO
    scgwo_weight_pop   = 20,
    scgwo_weight_iters = 50,
    scgwo_fs_pop       = 30,
    scgwo_fs_iters     = 100,

    epochs_cls      = 80,
    lr_cls          = 5e-5,
    patience_cls    = 10,
    batch_size_cls  = 32,

    max_feature_samples = 2000,
)

# NOTE: the epoch/iteration counts above match Tables 7-8 of the paper and
# are appropriate for a real GPU run. If you're doing a first quick sanity
# check on CPU, temporarily lower epochs_seg/epochs_cls and the scgwo_*_iters
# values (e.g. to 2-5) so the notebook finishes in a few minutes; then reset
# them to the values above for the real run whose results you'll report.

args = SimpleNamespace(**CONFIG)
os.makedirs(args.out_dir, exist_ok=True)
print("device:", args.device)
print("data_root:", args.data_root)


## 1. Pre-Processing — Section 3.1, Eq. (1)-(4)

Adaptive bilateral-Gaussian filtering (Eq. 1-3) and min-max normalisation
(Eq. 4).

In [ ]:
import torch
import torch.nn.functional as F


def _gaussian_kernel1d(sigma: float, radius: int, device, dtype) -> torch.Tensor:
    x = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
    k = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def gaussian_blur(x: torch.Tensor, sigma_g: float) -> torch.Tensor:
    """Eq. (1): d_bar = G_sigma_g * F, separable Gaussian convolution."""
    assert x.dim() == 4, "expected (B, C, H, W)"
    radius = max(1, int(3 * sigma_g))
    k1d = _gaussian_kernel1d(sigma_g, radius, x.device, x.dtype)
    kx = k1d.view(1, 1, 1, -1)
    ky = k1d.view(1, 1, -1, 1)
    c = x.shape[1]
    kx = kx.expand(c, 1, 1, kx.shape[-1])
    ky = ky.expand(c, 1, ky.shape[-2], 1)
    pad = radius
    x = F.conv2d(x, kx, padding=(0, pad), groups=c)
    x = F.conv2d(x, ky, padding=(pad, 0), groups=c)
    return x


def adaptive_bilateral_gaussian_filter(
    F_img: torch.Tensor,
    sigma_g: float = 2.0,
    sigma_s: float = 1.5,
    window: int = 5,
) -> torch.Tensor:
    """
    Eq. (1)-(3): adaptive bilateral-Gaussian filter.

    F_img : (B, 1, H, W) raw (noisy) slice, any real-valued range
    sigma_g : bandwidth of the Gaussian guidance blur (Eq. 1)
    sigma_s : joint spatial/range bandwidth sigma_s in Eq. (3)
    window  : neighbourhood window size (must be odd)

    Returns k(x), Eq. (2), same shape as F_img.
    """
    assert F_img.dim() == 4
    B, C, H, W = F_img.shape
    r = window // 2

    # Eq. (1): low-pass guidance image, decoupled from the noisy input
    d_bar = gaussian_blur(F_img, sigma_g)

    # Unfold into local (window x window) neighbourhoods for the weighted sum
    F_pad = F.pad(F_img, (r, r, r, r), mode="reflect")
    d_pad = F.pad(d_bar, (r, r, r, r), mode="reflect")

    F_unf = F_pad.unfold(2, window, 1).unfold(3, window, 1)  # (B,C,H,W,win,win)
    d_unf = d_pad.unfold(2, window, 1).unfold(3, window, 1)

    # spatial term: exp(-||x-y||^2 / sigma_s^2)
    yy, xx = torch.meshgrid(
        torch.arange(-r, r + 1, device=F_img.device, dtype=F_img.dtype),
        torch.arange(-r, r + 1, device=F_img.device, dtype=F_img.dtype),
        indexing="ij",
    )
    spatial_sq = (xx ** 2 + yy ** 2).view(1, 1, 1, 1, window, window)
    spatial_term = torch.exp(-spatial_sq / (sigma_s ** 2))

    # range term: exp(-||F_x - d_y||^2 / sigma_s^2), guidance-based (Eq. 3)
    F_center = F_img.unsqueeze(-1).unsqueeze(-1)  # (B,C,H,W,1,1)
    range_sq = (F_center - d_unf) ** 2
    range_term = torch.exp(-range_sq / (sigma_s ** 2))

    weight = spatial_term * range_term
    weight = weight / (weight.sum(dim=(-1, -2), keepdim=True) + 1e-8)  # L_x normaliser

    k_x = (weight * F_unf).sum(dim=(-1, -2))  # Eq. (2)
    return k_x


def min_max_normalize(a: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Eq. (4): a' = (a - min(A)) / (max(A) - min(A)), per-sample over (C,H,W)."""
    assert a.dim() == 4
    flat = a.flatten(1)
    a_min = flat.min(dim=1, keepdim=True).values
    a_max = flat.max(dim=1, keepdim=True).values
    a_norm = (flat - a_min) / (a_max - a_min + eps)
    return a_norm.view_as(a)


def preprocess_slice(
    raw_slice: torch.Tensor,
    sigma_g: float = 2.0,
    sigma_s: float = 1.5,
    window: int = 5,
) -> torch.Tensor:
    """Full Section 3.1 pipeline: Eq. (1)-(4) applied in sequence."""
    filtered = adaptive_bilateral_gaussian_filter(raw_slice, sigma_g, sigma_s, window)
    return min_max_normalize(filtered)


## 2. Swin-Residual Segmentation Network — Section 3.2, Eq. (5)-(15)

Patch embedding (Eq. 5), windowed self-attention (Eq. 6), shifted-window
transformer blocks (Eq. 7-8), patch merging (Eq. 9), residual decoder
(Eq. 12-13), and the Dice + boundary segmentation loss (Eq. 14-15).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------
# 3.2.2 Patch embedding (Eq. 5)
# --------------------------------------------------------------------------
class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, in_chans=1, embed_dim=48):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (B, C, H, W) -> tokens: (B, H/p * W/p, embed_dim)
        x = self.proj(x)  # Eq. (5): Reshape(a') . W_e + b_e, via strided conv
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, N, C)
        x = self.norm(x)
        return x, H, W


# --------------------------------------------------------------------------
# 3.2.3 Window multi-head self-attention (Eq. 6) with relative position bias
# --------------------------------------------------------------------------
def window_partition(x, H, W, window_size):
    B, N, C = x.shape
    x = x.view(B, H, W, C)
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, C)
    return windows


def window_reverse(windows, H, W, window_size, B):
    C = windows.shape[-1]
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, C)
    return x.view(B, H * W, C)


class WindowAttention(nn.Module):
    """Eq. (6): Attention(Q,K,V) = softmax(QK^T / sqrt(d) + B) V"""

    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

        # learnable relative position bias table, indexed by relative (dy,dx)
        self.rel_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        coords = torch.stack(torch.meshgrid(
            torch.arange(window_size), torch.arange(window_size), indexing="ij"))
        coords_flat = coords.flatten(1)
        rel_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
        rel_coords = rel_coords.permute(1, 2, 0).contiguous()
        rel_coords[:, :, 0] += window_size - 1
        rel_coords[:, :, 1] += window_size - 1
        rel_coords[:, :, 0] *= 2 * window_size - 1
        rel_index = rel_coords.sum(-1)
        self.register_buffer("rel_index", rel_index)
        nn.init.trunc_normal_(self.rel_bias_table, std=0.02)

    def forward(self, x, mask=None):
        Bw, N, C = x.shape
        qkv = self.qkv(x).reshape(Bw, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale  # QK^T / sqrt(d)

        bias = self.rel_bias_table[self.rel_index.view(-1)]
        bias = bias.view(N, N, -1).permute(2, 0, 1)  # B term in Eq. (6)
        attn = attn + bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(Bw // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(Bw, N, C)
        return self.proj(out)


# --------------------------------------------------------------------------
# 3.2.4 W-MSA / SW-MSA Swin block (Eq. 7-8)
# --------------------------------------------------------------------------
class SwinBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def _attn_mask(self, H, W, device):
        if self.shift_size == 0:
            return None
        img_mask = torch.zeros((1, H, W, 1), device=device)
        h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        mask_windows = window_partition(img_mask.flatten(1, 2), H, W, self.window_size)
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        return attn_mask

    def forward(self, x, H, W):
        B, N, C = x.shape
        shortcut = x
        x = self.norm1(x)  # Eq. (7)/(8): LN(z^{l-1})
        x = x.view(B, H, W, C)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        x_windows = window_partition(x.view(B, H * W, C), H, W, self.window_size)
        mask = self._attn_mask(H, W, x.device)
        attn_windows = self.attn(x_windows, mask=mask)
        x = window_reverse(attn_windows, H, W, self.window_size, B)
        x = x.view(B, H, W, C)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = x.view(B, H * W, C)
        x = shortcut + x                      # Eq. (7)/(8): + z^{l-1}
        x = x + self.mlp(self.norm2(x))        # Eq. (7)/(8): MLP(LN(.)) + .
        return x


# --------------------------------------------------------------------------
# Patch merging (Eq. 9)
# --------------------------------------------------------------------------
class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)  # Eq. (9): Concat of 2x2 neighbourhood
        x = x.view(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)                    # Eq. (9): Linear(.)
        return x, H // 2, W // 2


# --------------------------------------------------------------------------
# Four-stage hierarchical Swin encoder (Section 3.2.4)
# --------------------------------------------------------------------------
class SwinEncoder(nn.Module):
    def __init__(self, in_chans=1, embed_dim=48, depths=(2, 2, 2, 2),
                 num_heads=(2, 4, 8, 16), window_size=7, patch_size=4):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, in_chans, embed_dim)
        self.window_size = window_size
        self.stages = nn.ModuleList()
        self.merges = nn.ModuleList()
        dim = embed_dim
        for i, depth in enumerate(depths):
            blocks = nn.ModuleList([
                SwinBlock(dim, num_heads[i], window_size,
                          shift_size=0 if j % 2 == 0 else window_size // 2)
                for j in range(depth)
            ])
            self.stages.append(blocks)
            if i < len(depths) - 1:
                self.merges.append(PatchMerging(dim))
                dim *= 2
            else:
                self.merges.append(None)
        self.out_dims = [embed_dim * (2 ** i) for i in range(len(depths))]

    def forward(self, x):
        tokens, H, W = self.patch_embed(x)
        feats = []
        for blocks, merge in zip(self.stages, self.merges):
            for blk in blocks:
                tokens = blk(tokens, H, W)
            feats.append((tokens, H, W))  # store pre-merge feature for skip connections
            if merge is not None:
                tokens, H, W = merge(tokens, H, W)
        return feats  # list of (tokens, H, W) per stage, low-res last


# --------------------------------------------------------------------------
# 3.2.6 Residual convolutional decoder (Eq. 12-13)
# --------------------------------------------------------------------------
class ResidualBlock(nn.Module):
    """Eq. (12): R(x) = x + Phi(x, {W_i})"""

    def __init__(self, channels):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.phi(x))


class DecoderStage(nn.Module):
    """Eq. (13): D_s = R([Up(D_{s+1}) ; E_s])"""

    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.reduce = nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=1)
        self.res = ResidualBlock(out_ch)

    def forward(self, d_next, e_skip):
        d = self.up(d_next)
        if d.shape[-2:] != e_skip.shape[-2:]:
            d = F.interpolate(d, size=e_skip.shape[-2:], mode="bilinear", align_corners=False)
        fused = torch.cat([d, e_skip], dim=1)
        fused = self.reduce(fused)
        return self.res(fused)


def tokens_to_map(tokens, H, W):
    B, N, C = tokens.shape
    return tokens.transpose(1, 2).reshape(B, C, H, W)


class SwinResSegNet(nn.Module):
    """Full Section 3.2 model: Swin encoder + residual decoder -> tumour mask."""

    def __init__(self, in_chans=1, embed_dim=48, depths=(2, 2, 2, 2),
                 num_heads=(2, 4, 8, 16), window_size=7, patch_size=4, num_classes=1):
        super().__init__()
        self.encoder = SwinEncoder(in_chans, embed_dim, depths, num_heads, window_size, patch_size)
        dims = self.encoder.out_dims  # e.g. [48, 96, 192, 384]

        self.decoders = nn.ModuleList([
            DecoderStage(dims[3], dims[2], dims[2]),
            DecoderStage(dims[2], dims[1], dims[1]),
            DecoderStage(dims[1], dims[0], dims[0]),
        ])
        self.final_up = nn.ConvTranspose2d(dims[0], dims[0] // 2, kernel_size=patch_size, stride=patch_size)
        self.head = nn.Conv2d(dims[0] // 2, num_classes, kernel_size=1)

    def forward(self, x):
        feats = self.encoder(x)  # 4 stages, low-res last
        maps = [tokens_to_map(t, H, W) for (t, H, W) in feats]

        d = maps[3]
        d = self.decoders[0](d, maps[2])
        d = self.decoders[1](d, maps[1])
        d = self.decoders[2](d, maps[0])
        d = self.final_up(d)
        logits = self.head(d)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return torch.sigmoid(logits)  # tumour probability mask, Section 3.2.6


# --------------------------------------------------------------------------
# 3.2.7 Segmentation loss: Eq. (14)-(15)
# --------------------------------------------------------------------------
def dice_loss(pred, target, eps=1e-6):
    pred = pred.flatten(1)
    target = target.flatten(1)
    inter = (pred * target).sum(1)
    union = pred.sum(1) + target.sum(1)
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def boundary_loss_approx(pred, target):
    """
    Differentiable proxy for Eq. (15): penalises disagreement between predicted
    and ground-truth Sobel-gradient boundary maps (a smooth stand-in for the
    exact point-set Hausdorff-style boundary distance, which is non-differentiable).
    """
    def sobel_edges(m):
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=m.dtype, device=m.device).view(1, 1, 3, 3)
        ky = kx.transpose(-1, -2)
        gx = F.conv2d(m, kx, padding=1)
        gy = F.conv2d(m, ky, padding=1)
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-6)

    edge_pred = sobel_edges(pred)
    edge_true = sobel_edges(target)
    return F.l1_loss(edge_pred, edge_true)


def segmentation_loss(pred, target, lam=0.5):
    """Eq. (14): L_seg = DiceLoss + lambda * L_boundary."""
    return dice_loss(pred, target) + lam * boundary_loss_approx(pred, target)


## 3. Sine-Cosine Grey Wolf Optimizer — Section 3.3, Algorithm 1, Eq. (16)-(23)

Generic optimizer used both for Swin-ResSegNet weight refinement (Section
3.3) and, after sigmoid-thresholding, for wrapper feature selection
(Section 3.4.2).

In [ ]:
from dataclasses import dataclass
from typing import Callable, Optional
import numpy as np


@dataclass
class SCGWOConfig:
    population_size: int = 20       # N, Table 8
    iterations: int = 50            # T, Table 8
    p_min: float = 0.1              # Eq. (22)
    p_max: float = 0.5              # Eq. (22)
    a_max: float = 2.0              # Eq. (21), also GWO's own decay range
    lower_bound: float = -1.0
    upper_bound: float = 1.0
    seed: Optional[int] = None


class SCGWO:
    """Sine-Cosine Grey Wolf Optimizer (Algorithm 1)."""

    def __init__(self, dim: int, fitness_fn: Callable[[np.ndarray], float], config: SCGWOConfig = SCGWOConfig()):
        self.dim = dim
        self.fitness_fn = fitness_fn
        self.cfg = config
        self.rng = np.random.default_rng(config.seed)

        lo, hi = config.lower_bound, config.upper_bound
        self.population = self.rng.uniform(lo, hi, size=(config.population_size, dim))
        self.fitness = np.array([self.fitness_fn(x) for x in self.population])

        self.history = []  # best fitness per iteration, for convergence plots

    def _leaders(self):
        order = np.argsort(self.fitness)  # lower fitness = better
        alpha, beta, delta = self.population[order[0]], self.population[order[1]], self.population[order[2]]
        return alpha, beta, delta

    def _grey_wolf_update(self, x, leader, a):
        """Eq. (16)-(17) for a single leader."""
        r1 = self.rng.uniform(0, 1, size=self.dim)
        r2 = self.rng.uniform(0, 1, size=self.dim)
        A = 2 * a * r1 - a
        C = 2 * r2
        D = np.abs(C * leader - x)          # Eq. (16)
        return leader - A * D               # Eq. (17)

    def _sine_cosine_update(self, x, x_star, r1):
        """Eq. (19)-(20)."""
        r2 = self.rng.uniform(0, 2 * np.pi, size=self.dim)
        r3 = self.rng.uniform(0, 2, size=self.dim)
        r4 = self.rng.uniform(0, 1)
        if r4 < 0.5:
            return x + r1 * np.sin(r2) * np.abs(r3 * x_star - x)   # Eq. (19)
        else:
            return x + r1 * np.cos(r2) * np.abs(r3 * x_star - x)   # Eq. (20)

    def step(self, t: int):
        T = self.cfg.iterations
        a = self.cfg.a_max - t * (self.cfg.a_max / T)                 # GWO linear decay
        r1 = self.cfg.a_max - t * (self.cfg.a_max / T)                # Eq. (21)
        p_sc = self.cfg.p_min + (self.cfg.p_max - self.cfg.p_min) * (1 - t / T)  # Eq. (22)

        alpha, beta, delta = self._leaders()
        new_population = np.empty_like(self.population)

        for i, x in enumerate(self.population):
            if self.rng.uniform(0, 1) < p_sc:
                new_x = self._sine_cosine_update(x, alpha, r1)        # Eq. (19)-(20)
            else:
                x1 = self._grey_wolf_update(x, alpha, a)
                x2 = self._grey_wolf_update(x, beta, a)
                x3 = self._grey_wolf_update(x, delta, a)
                new_x = (x1 + x2 + x3) / 3.0                          # Eq. (18)
            new_population[i] = np.clip(new_x, self.cfg.lower_bound, self.cfg.upper_bound)

        self.population = new_population
        self.fitness = np.array([self.fitness_fn(x) for x in self.population])
        best_idx = int(np.argmin(self.fitness))
        self.history.append(self.fitness[best_idx])
        return self.population[best_idx], self.fitness[best_idx]

    def run(self):
        best_x, best_f = None, np.inf
        for t in range(1, self.cfg.iterations + 1):
            x, f = self.step(t)
            if f < best_f:
                best_x, best_f = x.copy(), f
        return best_x, best_f


## 4. VGG-16 Feature Extraction — Section 3.4.1, Eq. (24)

On Kaggle, `pretrained=True` will download ImageNet weights (internet
access is available in Kaggle notebooks when the "Internet" toggle in the
notebook settings is turned on).

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


class VGG16FeatureExtractor(nn.Module):
    def __init__(self, pretrained: bool = False, in_chans: int = 1):
        super().__init__()
        try:
            weights = models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
            vgg = models.vgg16(weights=weights)
        except Exception as e:  # offline sandbox: no internet access to fetch weights
            print(f"[vgg_feature_extractor] could not load pretrained weights ({e}); "
                  f"falling back to randomly-initialised VGG-16 (architecture only).")
            vgg = models.vgg16(weights=None)

        # VGG-16 expects 3-channel input; adapt the first conv layer for
        # single-channel (e.g. T1ce or FLAIR) MRI input if needed.
        if in_chans != 3:
            old_conv = vgg.features[0]
            new_conv = nn.Conv2d(in_chans, old_conv.out_channels,
                                  kernel_size=old_conv.kernel_size,
                                  stride=old_conv.stride, padding=old_conv.padding)
            with torch.no_grad():
                new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True).repeat(1, in_chans, 1, 1)
            vgg.features[0] = new_conv

        self.features = vgg.features
        self.avgpool = vgg.avgpool
        # keep only up to the penultimate FC layer (4096-dim, pre-classification), Eq. (24)
        self.fc = nn.Sequential(*list(vgg.classifier.children())[:-1])

    def forward(self, x):
        # x: (B, in_chans, H, W); VGG-16 expects >=224x224 input
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        v = self.fc(x)  # Eq. (24): v in R^4096
        return v


## 5. SC-GWO-Driven Feature Selection — Section 3.4.2, Algorithm 2, Eq. (25)-(26)

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split



def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def decode_mask(x: np.ndarray) -> np.ndarray:
    """Eq. (25): continuous SC-GWO position -> binary feature-selection mask."""
    return (sigmoid(x) >= 0.5).astype(np.float32)


def apply_mask(v: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Eq. (26): R = v elementwise* s."""
    return v * mask


def make_fitness_fn(v_train, y_train, v_val, y_val, alpha=0.99, k=5):
    """
    Builds the Eq. (23) fitness function for wrapper feature selection.
    v_train/v_val: (n_samples, 4096) raw VGG-16 feature vectors (Eq. 24)
    y_train/y_val: (n_samples,) binary labels (0=benign, 1=malignant)
    """
    beta = 1.0 - alpha
    n_features = v_train.shape[1]

    def fitness(x: np.ndarray) -> float:
        mask = decode_mask(x)
        if mask.sum() == 0:
            return 1.0  # degenerate empty-subset case: worst-possible fitness
        Rtr = apply_mask(v_train, mask[None, :])
        Rval = apply_mask(v_val, mask[None, :])
        # only keep the actually-selected columns for the k-NN fit (cheaper + correct)
        idx = np.where(mask > 0)[0]
        clf = KNeighborsClassifier(n_neighbors=min(k, len(y_train)))
        clf.fit(Rtr[:, idx], y_train)
        err = 1.0 - clf.score(Rval[:, idx], y_val)
        cardinality_term = mask.sum() / n_features
        return alpha * err + beta * cardinality_term  # Eq. (23)

    return fitness


def select_features(v_train, y_train, v_val, y_val, cfg: SCGWOConfig = None):
    """
    Algorithm 2, full procedure.
    Returns: (binary_mask, selected_feature_count, best_fitness, optimizer_history)
    """
    n_features = v_train.shape[1]
    if cfg is None:
        cfg = SCGWOConfig(population_size=30, iterations=100,
                           lower_bound=-4, upper_bound=4, seed=0)

    fitness_fn = make_fitness_fn(v_train, y_train, v_val, y_val)
    opt = SCGWO(dim=n_features, fitness_fn=fitness_fn, config=cfg)
    best_x, best_f = opt.run()
    mask = decode_mask(best_x)
    return mask, int(mask.sum()), best_f, opt.history


## 6. Wavelet-Frequency Attention Residual Network — Section 3.5, Eq. (27)-(35)

Discrete wavelet decomposition + attention gate (Eq. 27-29), spatial
multi-head attention (Eq. 30-31), fusion (Eq. 32), dense residual
classification head (Eq. 33-34), and the classification loss (Eq. 35).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pywt
import numpy as np


# --------------------------------------------------------------------------
# 3.5.2 Discrete wavelet decomposition branch (Eq. 27-29)
# --------------------------------------------------------------------------
def dwt_decompose(x: torch.Tensor, wavelet: str = "haar"):
    """
    Eq. (27): {I_LL, I_LH, I_HL, I_HH} = DWT(I_seg)
    x: (B, 1, H, W) single-channel tumour patch.
    Returns four (B, 1, H/2, W/2) sub-band tensors. Uses PyWavelets on the
    CPU/numpy side (non-differentiable through the DWT itself, matching the
    fixed-basis wavelet transform described in Section 3.5.2); the learnable
    part of the branch is the attention gate applied afterwards (Eq. 28-29).
    """
    B = x.shape[0]
    device, dtype = x.device, x.dtype
    x_np = x.detach().cpu().numpy()
    LL_list, LH_list, HL_list, HH_list = [], [], [], []
    for b in range(B):
        img = x_np[b, 0]
        LL, (LH, HL, HH) = pywt.dwt2(img, wavelet)
        LL_list.append(LL); LH_list.append(LH); HL_list.append(HL); HH_list.append(HH)
    to_t = lambda arr: torch.tensor(np.stack(arr), dtype=dtype, device=device).unsqueeze(1)
    return to_t(LL_list), to_t(LH_list), to_t(HL_list), to_t(HH_list)


class SubBandStem(nn.Module):
    """Lightweight convolutional stem applied to each DWT sub-band independently."""

    def __init__(self, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class WaveletAttentionBranch(nn.Module):
    """Eq. (28)-(29): learnable attention gate over the four DWT sub-bands."""

    def __init__(self, feat_dim=64, wavelet="haar"):
        super().__init__()
        self.wavelet = wavelet
        self.stem = SubBandStem(feat_dim)
        self.gate = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.Tanh(), nn.Linear(feat_dim, 1))
        self.feat_dim = feat_dim

    def forward(self, x):
        LL, LH, HL, HH = dwt_decompose(x, self.wavelet)
        f_LL, f_LH, f_HL, f_HH = (self.stem(b) for b in (LL, LH, HL, HH))
        stacked = torch.stack([f_LL, f_LH, f_HL, f_HH], dim=1)  # (B, 4, feat_dim)
        scores = self.gate(stacked).squeeze(-1)                  # (B, 4)
        alpha = torch.softmax(scores, dim=1)                     # Eq. (28)
        F_freq = (alpha.unsqueeze(-1) * stacked).sum(dim=1)       # Eq. (29)
        return F_freq, alpha


# --------------------------------------------------------------------------
# 3.5.3 Spatial multi-head attention branch (Eq. 30-31)
# --------------------------------------------------------------------------
class SpatialMultiHeadAttentionBranch(nn.Module):
    def __init__(self, in_chans=1, feat_dim=64, num_heads=8, tokens=16):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_chans, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, feat_dim, 3, stride=2, padding=1), nn.BatchNorm2d(feat_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(int(tokens ** 0.5)),
        )
        self.mha = nn.MultiheadAttention(embed_dim=feat_dim, num_heads=num_heads, batch_first=True)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        feat_map = self.backbone(x)                     # (B, C, h, w)
        B, C, h, w = feat_map.shape
        tokens = feat_map.flatten(2).transpose(1, 2)     # (B, N, C) -> Q=K=V input
        attn_out, _ = self.mha(tokens, tokens, tokens)   # Eq. (30)-(31)
        F_spatial = self.pool(attn_out.transpose(1, 2)).squeeze(-1)  # (B, C)
        return F_spatial


# --------------------------------------------------------------------------
# 3.5.5 Dense residual classification head (Eq. 32-34)
# --------------------------------------------------------------------------
class DenseResidualBlock(nn.Module):
    """Eq. (33): x_l = H_l([x_0, ..., x_{l-1}])"""

    def __init__(self, in_dim, growth=32):
        super().__init__()
        self.h = nn.Sequential(nn.Linear(in_dim, growth), nn.BatchNorm1d(growth), nn.ReLU(inplace=True))

    def forward(self, x_concat):
        return self.h(x_concat)


class DenseResidualHead(nn.Module):
    def __init__(self, in_dim, num_blocks=6, growth=32, num_classes=2):
        super().__init__()
        self.blocks = nn.ModuleList()
        dim = in_dim
        for _ in range(num_blocks):
            self.blocks.append(DenseResidualBlock(dim, growth))
            dim += growth
        self.out = nn.Linear(dim, num_classes)

    def forward(self, x0):
        feats = [x0]
        for blk in self.blocks:
            concat = torch.cat(feats, dim=1)
            feats.append(blk(concat))
        final = torch.cat(feats, dim=1)
        return self.out(final)  # logits, softmax applied in loss / at inference


class WFAResNet(nn.Module):
    """
    Full Section 3.5 model.
    Inputs:
      I_seg    : (B, 1, H, W) segmented tumour patch
      R_selected : (B, d_R) SC-GWO-selected VGG-16 feature subset (Eq. 26); pass
                   the full v (Eq. 24) if feature selection has not been run yet.
    """

    def __init__(self, selected_feat_dim, freq_dim=64, spatial_dim=64,
                 spatial_heads=8, num_dense_blocks=6, dense_growth=32,
                 fusion_dim=1024, num_classes=2, wavelet="haar"):
        super().__init__()
        self.wavelet_branch = WaveletAttentionBranch(freq_dim, wavelet)
        self.spatial_branch = SpatialMultiHeadAttentionBranch(1, spatial_dim, spatial_heads)
        fusion_in = freq_dim + spatial_dim + selected_feat_dim
        self.fusion = nn.Linear(fusion_in, fusion_dim)   # Eq. (32)
        self.head = DenseResidualHead(fusion_dim, num_dense_blocks, dense_growth, num_classes)  # Eq. (33)-(34)

    def forward(self, I_seg, R_selected):
        F_freq, alpha = self.wavelet_branch(I_seg)         # Eq. (27)-(29)
        F_spatial = self.spatial_branch(I_seg)              # Eq. (30)-(31)
        fused_in = torch.cat([F_spatial, F_freq, R_selected], dim=1)
        F_fused = F.relu(self.fusion(fused_in))              # Eq. (32)
        logits = self.head(F_fused)                          # Eq. (33)-(34) (pre-softmax)
        return logits, alpha


def classification_loss(logits, target):
    """Eq. (35): L_cls = -sum_c y_c log(y_hat_c), via stable cross-entropy."""
    return F.cross_entropy(logits, target)


## 7. Evaluation Metrics — Section 4.6, Eq. (36)-(47)

Every metric reported in the paper's Tables 12-15: DSC, IoU/Jaccard,
Hausdorff distance, sensitivity/specificity, SSIM, accuracy/precision/
recall/F1, MSE/RMSE, and AUC.

In [ ]:
import numpy as np
from scipy.spatial.distance import directed_hausdorff
from skimage.metrics import structural_similarity as skimage_ssim


# ---- Segmentation overlap metrics: Eq. (36)-(37) ----
def dice_score(pred_mask: np.ndarray, gt_mask: np.ndarray, eps=1e-8) -> float:
    """Eq. (36): DSC = 2|P n G| / (|P| + |G|)."""
    p, g = pred_mask.astype(bool), gt_mask.astype(bool)
    inter = np.logical_and(p, g).sum()
    return float((2 * inter + eps) / (p.sum() + g.sum() + eps))


def iou_score(pred_mask: np.ndarray, gt_mask: np.ndarray, eps=1e-8) -> float:
    """Eq. (37): IoU = |P n G| / |P u G|. Jaccard index is numerically identical (Section 4.6)."""
    p, g = pred_mask.astype(bool), gt_mask.astype(bool)
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    return float((inter + eps) / (union + eps))


jaccard_index = iou_score  # Section 4.6: reported alongside IoU, numerically identical


# ---- Boundary accuracy: Eq. (38) ----
def hausdorff_distance(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """Eq. (38): bidirectional Hausdorff distance between mask boundaries."""
    p_pts = np.argwhere(pred_mask > 0)
    g_pts = np.argwhere(gt_mask > 0)
    if len(p_pts) == 0 or len(g_pts) == 0:
        return float("inf")
    d1 = directed_hausdorff(p_pts, g_pts)[0]
    d2 = directed_hausdorff(g_pts, p_pts)[0]
    return float(max(d1, d2))


# ---- Detection-rate metrics: Eq. (39)-(40) ----
def sensitivity_recall(tp, fn) -> float:
    """Eq. (39)."""
    return float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0


def specificity(tn, fp) -> float:
    """Eq. (40)."""
    return float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0


# ---- Perceptual similarity: Eq. (41) ----
def ssim(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """Eq. (41): Structural Similarity Index Measure (via skimage's implementation)."""
    p = pred_mask.astype(np.float64)
    g = gt_mask.astype(np.float64)
    data_range = max(p.max() - p.min(), g.max() - g.min(), 1e-8)
    return float(skimage_ssim(p, g, data_range=data_range))


# ---- Classification correctness: Eq. (42)-(44) ----
def confusion_counts(y_true, y_pred, positive_label=1):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    tn = int(np.sum((y_true != positive_label) & (y_pred != positive_label)))
    fp = int(np.sum((y_true != positive_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred != positive_label)))
    return tp, tn, fp, fn


def accuracy(tp, tn, fp, fn) -> float:
    """Eq. (42)."""
    total = tp + tn + fp + fn
    return float((tp + tn) / total) if total > 0 else 0.0


def precision(tp, fp) -> float:
    """Eq. (43)."""
    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0


def f1_score(prec, rec) -> float:
    """Eq. (44)."""
    return float(2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0


# ---- Probability error: Eq. (45)-(46) ----
def mse(y_true, y_pred_prob) -> float:
    """Eq. (45)."""
    y_true, y_pred_prob = np.asarray(y_true, dtype=float), np.asarray(y_pred_prob, dtype=float)
    return float(np.mean((y_true - y_pred_prob) ** 2))


def rmse(y_true, y_pred_prob) -> float:
    """Eq. (46)."""
    return float(np.sqrt(mse(y_true, y_pred_prob)))


# ---- Discrimination ability: Eq. (47) ----
def auc_trapezoidal(y_true, y_score, n_thresholds=100) -> float:
    """Eq. (47): AUC via the trapezoidal rule over empirically swept thresholds."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    thresholds = np.linspace(1.0, 0.0, n_thresholds)
    tprs, fprs = [], []
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        tp, tn, fp, fn = confusion_counts(y_true, pred)
        tprs.append(tp / P if P > 0 else 0.0)
        fprs.append(fp / N if N > 0 else 0.0)
    auc = 0.0
    for k in range(len(thresholds) - 1):
        auc += (fprs[k + 1] - fprs[k]) * (tprs[k + 1] + tprs[k]) / 2.0
    return float(auc)


def full_segmentation_report(pred_mask, gt_mask) -> dict:
    return {
        "DSC": dice_score(pred_mask, gt_mask),
        "IoU": iou_score(pred_mask, gt_mask),
        "Jaccard": jaccard_index(pred_mask, gt_mask),
        "HausdorffDistance": hausdorff_distance(pred_mask, gt_mask),
        "SSIM": ssim(pred_mask, gt_mask),
    }


def full_classification_report(y_true, y_pred, y_score) -> dict:
    tp, tn, fp, fn = confusion_counts(y_true, y_pred)
    prec = precision(tp, fp)
    rec = sensitivity_recall(tp, fn)
    return {
        "Accuracy": accuracy(tp, tn, fp, fn),
        "Precision": prec,
        "Recall_Sensitivity": rec,
        "Specificity": specificity(tn, fp),
        "F1": f1_score(prec, rec),
        "MSE": mse(y_true, y_score),
        "RMSE": rmse(y_true, y_score),
        "AUC": auc_trapezoidal(y_true, y_score),
    }


In [ ]:

# The training pipeline functions below (Section 10, ported from
# train_full_pipeline.py) call metrics via an `M.<function>` module alias
# (as in the standalone script's `import metrics as M`). Since this notebook
# defines the metrics functions directly in the global namespace rather than
# as a separate module, this small shim reconstructs that `M.` interface.
import types as _types
M = _types.SimpleNamespace(
    dice_score=dice_score, iou_score=iou_score, jaccard_index=jaccard_index,
    hausdorff_distance=hausdorff_distance, sensitivity_recall=sensitivity_recall,
    specificity=specificity, ssim=ssim, confusion_counts=confusion_counts,
    accuracy=accuracy, precision=precision, f1_score=f1_score, mse=mse, rmse=rmse,
    auc_trapezoidal=auc_trapezoidal, full_segmentation_report=full_segmentation_report,
    full_classification_report=full_classification_report,
)
print("Metrics namespace ready.")


## 8. BraTS Dataset Loading — Section 4.1-4.2, Tables 4-5

Auto-detects either Kaggle folder convention, applies the patient-level,
HGG/LGG-stratified 80/20 split with a 15% validation hold-out (Table 5),
and extracts tumour-bearing axial slices (T1ce + FLAIR input).

In [ ]:
import os
import re
import glob
import csv
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import torch
from torch.utils.data import Dataset

try:
    import h5py
except ImportError:
    h5py = None

try:
    import nibabel as nib
except ImportError:
    nib = None


# ==========================================================================
# Format detection
# ==========================================================================
def detect_format(root_dir: str) -> str:
    """Returns 'h5' or 'nifti' depending on what's found under root_dir."""
    for _, _, files in os.walk(root_dir):
        if any(re.match(r"volume_\d+_slice_\d+\.h5$", f) for f in files):
            return "h5"
        if any(f.endswith("_seg.nii.gz") or f.endswith("_seg.nii") for f in files):
            return "nifti"
    raise RuntimeError(
        f"Could not detect a supported BraTS format under {root_dir}. "
        f"Expected either 'volume_N_slice_S.h5' files (Kaggle H5 format) "
        f"or '*_seg.nii(.gz)' files (raw NIfTI format)."
    )


# ==========================================================================
# FORMAT A: H5-per-slice dataset
# ==========================================================================
class BraTSH5SliceDataset(Dataset):
    """
    Wraps a list of volume_N_slice_S.h5 file paths. Each file already IS a
    single 2D slice (no further slice-extraction needed, unlike Format B).

    image  : (240, 240, 4) -> returned as (2, H, W) using channels
             [T1ce_index, FLAIR_index] per `input_channel_indices`
    mask   : (240, 240, 3) one-hot sub-region channels -> collapsed to a
             single binary whole-tumour mask (any channel > 0)
    label  : always -1 (no grade information in this format; see module
             docstring). Classification cannot be trained from this format
             alone.
    """

    # NOTE: verify these indices against your own diagnostic cell output
    # (`f['image'].shape`, and inspect a slice's channel-wise appearance)
    # before trusting them blindly -- different re-uploads of this dataset
    # have been observed with different channel orderings. The common
    # convention for this specific Kaggle dataset is
    # image channels = [T1, T1ce, T2, FLAIR] (indices 0,1,2,3).
    T1CE_CHANNEL = 1
    FLAIR_CHANNEL = 3

    def __init__(self, h5_paths: List[str]):
        self.h5_paths = h5_paths

    def __len__(self):
        return len(self.h5_paths)

    def __getitem__(self, idx):
        path = self.h5_paths[idx]
        with h5py.File(path, "r") as f:
            image = f["image"][:].astype(np.float32)  # (240, 240, 4)
            mask = f["mask"][:].astype(np.float32)     # (240, 240, 3)

        t1ce = image[:, :, self.T1CE_CHANNEL]
        flair = image[:, :, self.FLAIR_CHANNEL]
        slice_input = np.stack([t1ce, flair], axis=0)  # (2, H, W)

        whole_tumour = (mask.sum(axis=-1) > 0).astype(np.float32)[None, :, :]  # (1, H, W)

        m = re.search(r"volume_(\d+)_slice_(\d+)\.h5$", os.path.basename(path))
        vol_id, slice_id = (m.group(1), m.group(2)) if m else ("?", "?")

        return (torch.from_numpy(slice_input).float(),
                torch.from_numpy(whole_tumour).float(),
                -1,  # no grade label available in this format
                {"patient_id": f"volume_{vol_id}", "slice_index": slice_id})


def _h5_patient_split(h5_paths: List[str], test_frac=0.2, val_frac_of_train=0.15, seed=0):
    """Volume-level (not slice-level) split, since grade-stratification is
    unavailable in this format; groups slices by their volume_N id."""
    by_volume = {}
    for p in h5_paths:
        m = re.search(r"volume_(\d+)_slice_\d+\.h5$", os.path.basename(p))
        vol = m.group(1) if m else "unknown"
        by_volume.setdefault(vol, []).append(p)

    vol_ids = sorted(by_volume.keys())
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(vol_ids))
    vol_ids = [vol_ids[i] for i in idx]

    n_test = int(round(len(vol_ids) * test_frac))
    test_vols = vol_ids[:n_test]
    train_vols_all = vol_ids[n_test:]
    n_val = int(round(len(train_vols_all) * val_frac_of_train))
    val_vols = train_vols_all[:n_val]
    train_vols = train_vols_all[n_val:]

    train = [p for v in train_vols for p in by_volume[v]]
    val = [p for v in val_vols for p in by_volume[v]]
    test = [p for v in test_vols for p in by_volume[v]]
    return train, val, test, len(vol_ids)


def build_datasets_h5(root_dir: str, seed: int = 0):
    h5_paths = []
    for r, _, files in os.walk(root_dir):
        h5_paths += [os.path.join(r, f) for f in files if re.match(r"volume_\d+_slice_\d+\.h5$", f)]
    if not h5_paths:
        raise RuntimeError(f"No volume_N_slice_S.h5 files found under {root_dir}")
    h5_paths.sort()

    train_p, val_p, test_p, n_vols = _h5_patient_split(h5_paths, seed=seed)
    print(f"[brats_dataset] H5 format detected. {n_vols} volumes, {len(h5_paths)} total slices.")
    print(f"[brats_dataset] split (volume-level): train={len(train_p)} slices, "
          f"val={len(val_p)} slices, test={len(test_p)} slices")
    print("[brats_dataset] NOTE: no grade (HGG/LGG) labels available in this format; "
          "all labels are -1. Classification (WFA-ResNet) cannot be trained from "
          "this dataset alone -- see the notebook markdown for options.")

    return (BraTSH5SliceDataset(train_p), BraTSH5SliceDataset(val_p), BraTSH5SliceDataset(test_p)), False


# ==========================================================================
# FORMAT B: raw per-patient NIfTI volumes (unchanged from the original loader)
# ==========================================================================
@dataclass
class PatientRecord:
    patient_id: str
    folder: str
    grade: Optional[str]
    sequences: dict


def _find_seq_file(folder: str, patient_id: str, suffix: str) -> Optional[str]:
    for ext in (".nii.gz", ".nii"):
        candidate = os.path.join(folder, f"{patient_id}_{suffix}{ext}")
        if os.path.isfile(candidate):
            return candidate
    matches = glob.glob(os.path.join(folder, f"*_{suffix}.nii*"))
    return matches[0] if matches else None


def _load_grade_mapping(root_dir: str) -> dict:
    mapping = {}
    csv_path = os.path.join(root_dir, "name_mapping.csv")
    if not os.path.isfile(csv_path):
        return mapping
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        grade_col = next((c for c in reader.fieldnames if c.strip().lower() == "grade"), None)
        id_cols = [c for c in reader.fieldnames if "subject_id" in c.lower()]
        for row in reader:
            grade = row.get(grade_col) if grade_col else None
            for id_col in id_cols:
                pid = row.get(id_col)
                if pid:
                    mapping[pid] = grade
    return mapping


def discover_patients_nifti(root_dir: str) -> List[PatientRecord]:
    records = []
    grade_map = _load_grade_mapping(root_dir)

    for grade in ("HGG", "LGG"):
        grade_dir = os.path.join(root_dir, grade)
        if os.path.isdir(grade_dir):
            for patient_folder in sorted(glob.glob(os.path.join(grade_dir, "*"))):
                if not os.path.isdir(patient_folder):
                    continue
                pid = os.path.basename(patient_folder)
                seqs = {s: _find_seq_file(patient_folder, pid, s) for s in ("t1", "t1ce", "t2", "flair", "seg")}
                if seqs["t1ce"] and seqs["flair"]:
                    records.append(PatientRecord(pid, patient_folder, grade, seqs))
    if records:
        return records

    for patient_folder in sorted(glob.glob(os.path.join(root_dir, "*"))):
        if not os.path.isdir(patient_folder):
            continue
        pid = os.path.basename(patient_folder)
        seqs = {s: _find_seq_file(patient_folder, pid, s) for s in ("t1", "t1ce", "t2", "flair", "seg")}
        if seqs["t1ce"] and seqs["flair"]:
            records.append(PatientRecord(pid, patient_folder, grade_map.get(pid), seqs))
    return records


def stratified_patient_split(records: List[PatientRecord], test_frac=0.2, val_frac_of_train=0.15, seed=0):
    rng = np.random.default_rng(seed)
    by_grade = {}
    for r in records:
        by_grade.setdefault(r.grade, []).append(r)

    train, val, test = [], [], []
    for grade, group in by_grade.items():
        idx = rng.permutation(len(group))
        group = [group[i] for i in idx]
        n_test = int(round(len(group) * test_frac))
        test_part = group[:n_test]
        train_part = group[n_test:]
        n_val = int(round(len(train_part) * val_frac_of_train))
        val.extend(train_part[:n_val])
        train.extend(train_part[n_val:])
        test.extend(test_part)
    return train, val, test


def extract_tumour_slices(record: PatientRecord, min_tumour_pixels: int = 50) -> List[int]:
    if record.sequences["seg"] is None:
        return []
    seg = nib.load(record.sequences["seg"]).get_fdata()
    counts = (seg > 0).sum(axis=(0, 1))
    return [i for i in range(seg.shape[2]) if counts[i] >= min_tumour_pixels]


class BraTSNiftiSliceDataset(Dataset):
    def __init__(self, records: List[PatientRecord], min_tumour_pixels: int = 50,
                 cache_slices: bool = True, max_cached_volumes: int = 6):
        self.records = records
        self.index = []
        self._cache_slices = cache_slices
        self._max_cached_volumes = max_cached_volumes
        self._volume_cache = {}   # bounded LRU: at most max_cached_volumes entries
        self._cache_order = []    # tracks access order for eviction
        for ri, rec in enumerate(records):
            for si in extract_tumour_slices(rec, min_tumour_pixels):
                self.index.append((ri, si))

    def __len__(self):
        return len(self.index)

    def _load_volume(self, rec: PatientRecord):
        if self._cache_slices and rec.patient_id in self._volume_cache:
            self._cache_order.remove(rec.patient_id)
            self._cache_order.append(rec.patient_id)  # mark as most-recently-used
            return self._volume_cache[rec.patient_id]
        t1ce = nib.load(rec.sequences["t1ce"]).get_fdata().astype(np.float32)
        flair = nib.load(rec.sequences["flair"]).get_fdata().astype(np.float32)
        seg = nib.load(rec.sequences["seg"]).get_fdata().astype(np.float32) if rec.sequences["seg"] else None
        vol = (t1ce, flair, seg)
        if self._cache_slices:
            self._volume_cache[rec.patient_id] = vol
            self._cache_order.append(rec.patient_id)
            # evict least-recently-used volumes to keep memory bounded
            while len(self._cache_order) > self._max_cached_volumes:
                oldest = self._cache_order.pop(0)
                self._volume_cache.pop(oldest, None)
        return vol

    def __getitem__(self, idx):
        ri, si = self.index[idx]
        rec = self.records[ri]
        t1ce, flair, seg = self._load_volume(rec)
        t1ce_slice, flair_slice = t1ce[:, :, si], flair[:, :, si]
        seg_slice = seg[:, :, si] if seg is not None else np.zeros_like(t1ce_slice)

        slice_input = np.stack([t1ce_slice, flair_slice], axis=0)
        mask = (seg_slice > 0).astype(np.float32)[None, :, :]

        label = 1 if rec.grade == "HGG" else (0 if rec.grade == "LGG" else -1)
        meta = {"patient_id": rec.patient_id, "slice_index": si}
        return (torch.from_numpy(slice_input).float(), torch.from_numpy(mask).float(), label, meta)


def build_datasets_nifti(root_dir: str, min_tumour_pixels: int = 50, seed: int = 0):
    records = discover_patients_nifti(root_dir)
    if not records:
        raise RuntimeError(f"No BraTS patient folders found under {root_dir}.")
    train_recs, val_recs, test_recs = stratified_patient_split(records, seed=seed)
    has_grades = any(r.grade is not None for r in records)
    print(f"[brats_dataset] NIfTI format detected. {len(records)} patients "
          f"({sum(r.grade == 'HGG' for r in records)} HGG, "
          f"{sum(r.grade == 'LGG' for r in records)} LGG, "
          f"{sum(r.grade is None for r in records)} unknown grade)")
    print(f"[brats_dataset] split: train={len(train_recs)}  val={len(val_recs)}  test={len(test_recs)}")

    ds = (BraTSNiftiSliceDataset(train_recs, min_tumour_pixels),
          BraTSNiftiSliceDataset(val_recs, min_tumour_pixels),
          BraTSNiftiSliceDataset(test_recs, min_tumour_pixels))
    return ds, has_grades


# ==========================================================================
# Unified entry point
# ==========================================================================
def build_datasets(root_dir: str, min_tumour_pixels: int = 50, seed: int = 0):
    """
    Auto-detects the dataset format under root_dir and returns
    (train_ds, val_ds, test_ds), has_grade_labels.

    has_grade_labels is False for the H5 format (Format A) and for any
    NIfTI dataset with no grade metadata found; in that case every sample's
    `label` is -1 and the classification stage (WFA-ResNet) should be
    skipped, or trained on a different, grade-labelled dataset.
    """
    fmt = detect_format(root_dir)
    if fmt == "h5":
        if h5py is None:
            raise ImportError("This dataset requires h5py. Install with: pip install h5py")
        (train_ds, val_ds, test_ds), has_grades = build_datasets_h5(root_dir, seed=seed)
    else:
        if nib is None:
            raise ImportError("This dataset requires nibabel. Install with: pip install nibabel")
        (train_ds, val_ds, test_ds), has_grades = build_datasets_nifti(root_dir, min_tumour_pixels, seed=seed)
    return train_ds, val_ds, test_ds, has_grades


# ==========================================================================
# Self-test: builds tiny, CORRECTLY-STRUCTURED synthetic trees for BOTH
# formats and confirms both code paths work against real file I/O.
# ==========================================================================


## 10. End-to-End Training Pipeline — Section 3.6, Algorithm 3

The functions below implement Algorithm 3's Phase A (training) line by
line: Swin-ResSegNet gradient pre-training (line 2), SC-GWO weight
refinement (line 3), segmentation + crop (line 4), VGG-16 feature
extraction + SC-GWO feature selection (lines 5-6), and WFA-ResNet training
(line 7). `evaluate()` implements Phase B on the held-out test partition
and produces the metrics for Tables 12-15.

In [ ]:
import argparse
import os
import json
import csv
import time

import numpy as np
import torch



def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


def resize_batch(x, size=224):
    return torch.nn.functional.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)


def resize_mask(x, size=224):
    """Resizes a binary mask using NEAREST-neighbour interpolation, which
    preserves exact 0/1 values (no blurred/fractional edge pixels). Using
    bilinear interpolation on a binary mask (as images correctly use)
    creates soft, non-binary edges that distort DSC/IoU/Jaccard and can
    make Hausdorff distance computation fail. Always use this function
    (never resize_batch) for ground-truth or predicted masks."""
    resized = torch.nn.functional.interpolate(x, size=(size, size), mode="nearest")
    return (resized >= 0.5).float()


# --------------------------------------------------------------------------
# Stage 1: Swin-ResSegNet gradient pre-training (Section 3.2.8, Table 7)
# --------------------------------------------------------------------------
def run_segmentation_training(train_loader, val_loader, args, device):
    model = SwinResSegNet(in_chans=1, embed_dim=48, depths=(2, 2, 2, 2),
                           num_heads=(2, 4, 8, 16), window_size=7).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr_seg, betas=(0.9, 0.999), weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs_seg, eta_min=1e-6)

    best_val_loss, best_state, patience_ctr = float("inf"), None, 0

    for epoch in range(1, args.epochs_seg + 1):
        model.train()
        train_losses = []
        for slice_input, mask, label, meta in train_loader:
            # slice_input: (B, 2, H, W) = [T1ce, FLAIR]; segmentation uses FLAIR (whole-tumour
            # contrast) as the primary channel, per Section 4.1's modality rationale.
            flair = slice_input[:, 1:2].to(device)
            flair = resize_batch(flair)
            flair = preprocess_slice(flair)  # Eq. (1)-(4)
            mask = resize_mask(mask.to(device))

            opt.zero_grad()
            pred = model(flair)
            loss = segmentation_loss(pred, mask)
            loss.backward()
            opt.step()
            train_losses.append(loss.item())
        sched.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for slice_input, mask, label, meta in val_loader:
                flair = resize_batch(slice_input[:, 1:2].to(device))
                flair = preprocess_slice(flair)
                mask = resize_mask(mask.to(device))
                pred = model(flair)
                val_losses.append(segmentation_loss(pred, mask).item())

        train_loss, val_loss = float(np.mean(train_losses)), float(np.mean(val_losses))
        print(f"[Swin-ResSegNet] epoch {epoch}/{args.epochs_seg}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, best_state, patience_ctr = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= args.patience_seg:
                print(f"[Swin-ResSegNet] early stopping at epoch {epoch} (patience={args.patience_seg})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# --------------------------------------------------------------------------
# Stage 2: SC-GWO weight refinement (Section 3.3, Table 8)
# --------------------------------------------------------------------------
def run_scgwo_weight_refinement(model, val_loader, args, device):
    # gather one fixed validation batch for fitness evaluation (Section 3.3.5)
    slice_input, mask, label, meta = next(iter(val_loader))
    flair = resize_batch(slice_input[:, 1:2].to(device))
    flair = preprocess_slice(flair)
    mask = resize_mask(mask.to(device))

    head = model.head
    base_w, base_b = head.weight.detach().clone(), head.bias.detach().clone()
    D = base_w.numel() + base_b.numel()

    def fitness(x):
        x_t = torch.tensor(x, dtype=base_w.dtype, device=device)
        with torch.no_grad():
            head.weight.copy_(base_w + 0.01 * x_t[: base_w.numel()].view_as(base_w))
            head.bias.copy_(base_b + 0.01 * x_t[base_w.numel():].view_as(base_b))
            pred = model(flair)
            return float(segmentation_loss(pred, mask).item())

    cfg = SCGWOConfig(population_size=args.scgwo_weight_pop, iterations=args.scgwo_weight_iters,
                       lower_bound=-1, upper_bound=1, seed=args.seed)
    opt = SCGWO(dim=D, fitness_fn=fitness, config=cfg)
    best_x, best_f = opt.run()
    print(f"[SC-GWO weight refinement] fitness {opt.history[0]:.4f} -> {best_f:.4f} over {len(opt.history)} iterations")

    with torch.no_grad():
        best_t = torch.tensor(best_x, dtype=base_w.dtype, device=device)
        head.weight.copy_(base_w + 0.01 * best_t[: base_w.numel()].view_as(base_w))
        head.bias.copy_(base_b + 0.01 * best_t[base_w.numel():].view_as(base_b))
    return model, opt.history


# --------------------------------------------------------------------------
# Stage 3: segment + crop + VGG-16 features + SC-GWO feature selection
# --------------------------------------------------------------------------
def build_feature_bank(seg_model, loader, vgg_model, device, max_samples=None):
    seg_model.eval(); vgg_model.eval()
    feats, labels, I_segs = [], [], []
    with torch.no_grad():
        for slice_input, mask, label, meta in loader:
            flair = resize_batch(slice_input[:, 1:2].to(device))
            flair_pp = preprocess_slice(flair)
            prob = seg_model(flair_pp)
            pred_mask = (prob >= 0.5).float()
            I_seg = flair_pp * pred_mask  # crop to predicted tumour region

            v = vgg_model(resize_batch(I_seg, 224)).cpu().numpy()
            for b in range(v.shape[0]):
                if int(label[b]) == -1:
                    continue  # skip unknown-grade cases for the classification stage
                feats.append(v[b])
                labels.append(int(label[b]))
                I_segs.append(I_seg[b:b + 1].cpu())
            if max_samples and len(feats) >= max_samples:
                break
    return np.stack(feats), np.array(labels), I_segs


# --------------------------------------------------------------------------
# Stage 4: WFA-ResNet training (Section 3.5.6, Table 7)
# --------------------------------------------------------------------------
def run_classification_training(I_segs_train, R_train, y_train, I_segs_val, R_val, y_val, args, device):
    model = WFAResNet(selected_feat_dim=R_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr_cls, betas=(0.9, 0.999), weight_decay=1e-5)

    I_train_t = torch.cat(I_segs_train, dim=0).to(device)
    R_train_t = torch.tensor(R_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
    I_val_t = torch.cat(I_segs_val, dim=0).to(device)
    R_val_t = torch.tensor(R_val, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val, dtype=torch.long, device=device)

    # FIX: class-weighted loss (inverse frequency) so the majority class (HGG) cannot dominate
    counts = np.bincount(y_train, minlength=2).astype(np.float32)
    cls_w = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1.0)), dtype=torch.float32, device=device)
    print(f"[WFA-ResNet] train class counts (benign, malignant) = {counts.tolist()}, "
          f"val class counts = {np.bincount(y_val, minlength=2).tolist()}, loss weights = {cls_w.tolist()}")
    best_val_loss, best_state, patience_ctr = float("inf"), None, 0
    n = I_train_t.shape[0]
    batch_size = min(args.batch_size_cls, n)

    for epoch in range(1, args.epochs_cls + 1):
        model.train()
        perm = torch.randperm(n)
        losses = []
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits, alpha = model(I_train_t[idx], R_train_t[idx])
            loss = F.cross_entropy(logits, y_train_t[idx], weight=cls_w)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            logits, _ = model(I_val_t, R_val_t)
            val_loss = F.cross_entropy(logits, y_val_t, weight=cls_w).item()

        print(f"[WFA-ResNet] epoch {epoch}/{args.epochs_cls}  train_loss={np.mean(losses):.4f}  val_loss={val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss, best_state, patience_ctr = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= args.patience_cls:
                print(f"[WFA-ResNet] early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# --------------------------------------------------------------------------
# Evaluation: produces the rows of Tables 12-15 for THIS SEED
# --------------------------------------------------------------------------
def evaluate_segmentation_only(seg_model, test_loader, device):
    """Same as evaluate() but skips the classification stage entirely, for
    datasets with no grade labels (has_grades == False)."""
    seg_model.eval()
    dsc_list, iou_list, hd_list = [], [], []
    with torch.no_grad():
        for slice_input, mask, label, meta in test_loader:
            flair = resize_batch(slice_input[:, 1:2].to(device))
            flair_pp = preprocess_slice(flair)
            gt_mask = resize_mask(mask.to(device))
            prob = seg_model(flair_pp)
            pred_mask = (prob >= 0.5).float()
            for b in range(pred_mask.shape[0]):
                pm = pred_mask[b, 0].cpu().numpy()
                gm = gt_mask[b, 0].cpu().numpy()
                dsc_list.append(M.dice_score(pm, gm))
                iou_list.append(M.iou_score(pm, gm))
                if pm.sum() > 0 and gm.sum() > 0:
                    hd_list.append(M.hausdorff_distance(pm, gm))

    if not hd_list:
        print(f"[evaluate] NOTE: Hausdorff distance is NaN because 0/{len(dsc_list)} test "
              f"samples had BOTH a non-empty predicted mask and a non-empty ground-truth mask "
              f"at the same time. This usually means the model is under-trained (e.g. very few "
              f"epochs) and is predicting empty masks for every test slice. Check that DSC/IoU "
              f"above are also low/zero -- if so, this is consistent (an under-trained model), "
              f"not a separate bug. Try more epochs (Table 7's epochs_seg=100) for a real run.")

    seg_report = {
        "DSC": float(np.mean(dsc_list)) if dsc_list else float("nan"),
        "IoU": float(np.mean(iou_list)) if iou_list else float("nan"),
        "Jaccard": float(np.mean(iou_list)) if iou_list else float("nan"),
        "HausdorffDistance": float(np.mean(hd_list)) if hd_list else float("nan"),
    }
    return seg_report, {}


def evaluate(seg_model, wfa_model, vgg_model, sel_idx, test_loader, device):
    seg_model.eval(); wfa_model.eval(); vgg_model.eval()
    dsc_list, iou_list, hd_list = [], [], []
    y_true, y_pred, y_score = [], [], []

    with torch.no_grad():
        for slice_input, mask, label, meta in test_loader:
            flair = resize_batch(slice_input[:, 1:2].to(device))
            flair_pp = preprocess_slice(flair)
            gt_mask = resize_mask(mask.to(device))
            prob = seg_model(flair_pp)
            pred_mask = (prob >= 0.5).float()

            for b in range(pred_mask.shape[0]):
                pm = pred_mask[b, 0].cpu().numpy()
                gm = gt_mask[b, 0].cpu().numpy()
                dsc_list.append(M.dice_score(pm, gm))
                iou_list.append(M.iou_score(pm, gm))
                if pm.sum() > 0 and gm.sum() > 0:
                    hd_list.append(M.hausdorff_distance(pm, gm))

            if int(label[0]) == -1:
                continue
            I_seg = flair_pp * pred_mask
            v = vgg_model(resize_batch(I_seg, 224)).cpu().numpy()
            R = v[:, sel_idx]
            logits, _ = wfa_model(I_seg, torch.tensor(R, dtype=torch.float32, device=device))
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            for b in range(len(label)):
                if int(label[b]) == -1:
                    continue
                y_true.append(int(label[b]))
                y_pred.append(int(preds[b]))
                y_score.append(float(probs[b]))

    seg_report = {
        "DSC": float(np.mean(dsc_list)) if dsc_list else float("nan"),
        "IoU": float(np.mean(iou_list)) if iou_list else float("nan"),
        "Jaccard": float(np.mean(iou_list)) if iou_list else float("nan"),
        "HausdorffDistance": float(np.mean(hd_list)) if hd_list else float("nan"),
    }
    cls_report = M.full_classification_report(y_true, y_pred, y_score) if y_true else {}
    if y_true:
        from sklearn.metrics import roc_auc_score
        try:
            cls_report["AUC_exact"] = float(roc_auc_score(y_true, y_score))  # rank-based, no threshold grid
        except ValueError:
            cls_report["AUC_exact"] = float("nan")
        cls_report["BalancedAccuracy"] = 0.5 * (cls_report["Recall_Sensitivity"] + cls_report["Specificity"])
        cls_report["test_malignant_fraction"] = float(np.mean(y_true))
    return seg_report, cls_report


# Full experiment runner (Sections 15-19)

**This notebook = the original pipeline (Sections 0-10) + everything the paper still needs:**
5-seed runs, patient-level metrics, SC-GWO ablations, classification baselines (CNN, DNN, ResNet, DenseNet, ResDenseNet),
segmentation baselines (Faster R-CNN, YOLO), timings, figures, significance tests.

**How to run**
1. Run the cells top to bottom (Run All). Everything you may need to edit is in the cell titled *15.0 Experiment settings*.
2. First set `PRESET = "quick"` (a smoke test, minutes) to check that every stage runs on your machine.
3. Then `PRESET = "single"` (seed 0, all tables) and, once happy, `PRESET = "full"` (seeds 0-4).
4. Results are saved per seed and **re-running skips finished seeds**, so you can continue across sessions.
5. Send back `ALL_RESULTS.zip` (last cell).

Fixes already included: shuffled validation bank, class-weighted loss, exact AUC, preloaded slices (fast data loading).


## 15.0 Experiment settings

In [ ]:
# ======================= 15.0 Experiment settings (EDIT HERE) =======================
PRESET = "single"           # "quick" = smoke test | "single" = seed 0, all tables | "full" = seeds 0-4
DATASET_NAME = "BraTS2019"  # label used in file names / tables
import os
# --- locate BOTH datasets automatically (Add Input: BraTS 2019 *and* BraTS2020 Dataset (Training + Validation)) ---
ROOTS = {}
for _dp, _dirs, _files in os.walk("/kaggle/input"):
    if "HGG" in _dirs and "LGG" in _dirs and "BraTS2019" not in ROOTS: ROOTS["BraTS2019"] = _dp
    if "name_mapping.csv" in _files and any(d.startswith("BraTS20_Training") for d in _dirs) and "BraTS2020" not in ROOTS: ROOTS["BraTS2020"] = _dp
print("datasets found:", ROOTS)
DATA_ROOT_OVERRIDE = ROOTS.get("BraTS2019") or ROOTS.get("BraTS2020")   # first dataset; the final cell loops over both
OUT_ROOT = "./results_full"
FORCE_RERUN = False         # True = ignore finished seeds and recompute
SEEDS_OVERRIDE = None       # e.g. [1, 2] = run only these seeds in this session (use different seeds in different sessions)
# ====================================================================================

import os, time, json, copy, math, random, glob, zipfile
from types import SimpleNamespace
import numpy as np
import torch

_BASE = dict(
    batch_size_seg=16, lr_seg=1e-4, lr_cls=5e-5, batch_size_cls=32, min_tumour_pixels=50,
    refine_batch=64,                 # slices (from several val patients) used as SC-GWO weight-refinement fitness batch
    base_lr_seg=5e-3, base_batch_seg=8, base_batch_cls=32,
    pretrained_baselines=True,       # ImageNet weights for ResNet/DenseNet/Faster R-CNN/YOLO (needs internet)
    base_train_cap=None,             # e.g. 8000 = train segmentation baselines on a random subset (faster); None = all slices
    max_patients=None,               # debug only: use a small stratified subset of patients
    run_baselines=True, run_ablation=True, run_yolo=True,
    yolo_models={"YOLO v9": "yolov9c-seg.pt", "YOLO v8": "yolov8m-seg.pt"},
)
PRESETS = {
    "quick": dict(seeds=[0], ablation_seeds=[0], epochs_seg=1, patience_seg=1,
                  scgwo_weight_pop=4, scgwo_weight_iters=2, scgwo_fs_pop=6, scgwo_fs_iters=3,
                  epochs_cls=2, patience_cls=2, max_feature_samples=96,
                  base_epochs_seg=1, base_epochs_cls=1, base_patience=1,
                  max_patients=16, batch_size_seg=8, refine_batch=16, pretrained_baselines=False, run_yolo=False),
    "single": dict(seeds=[0], ablation_seeds=[0], epochs_seg=30, patience_seg=6,
                   scgwo_weight_pop=20, scgwo_weight_iters=50, scgwo_fs_pop=30, scgwo_fs_iters=100,
                   epochs_cls=40, patience_cls=6, max_feature_samples=2000,
                   base_epochs_seg=5, base_epochs_cls=15, base_patience=4, base_train_cap=3000, run_yolo=True),
    "full": dict(seeds=[0, 1, 2, 3, 4], ablation_seeds=[0], epochs_seg=100, patience_seg=15,
                 scgwo_weight_pop=20, scgwo_weight_iters=50, scgwo_fs_pop=30, scgwo_fs_iters=100,
                 epochs_cls=80, patience_cls=10, max_feature_samples=2000,
                 base_epochs_seg=10, base_epochs_cls=30, base_patience=8, base_train_cap=6000),
}
cfg = SimpleNamespace(**{**_BASE, **PRESETS[PRESET]})
if SEEDS_OVERRIDE is not None: cfg.seeds = list(SEEDS_OVERRIDE)
DATA_ROOT = DATA_ROOT_OVERRIDE or AUTO_DATA_ROOT
assert DATA_ROOT, "Set DATA_ROOT_OVERRIDE above to your BraTS folder."
OUT_DIR = os.path.join(OUT_ROOT, DATASET_NAME, "quick" if PRESET == "quick" else "main")  # single & full share one folder
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = args.device
print(f"preset={PRESET}  seeds={cfg.seeds}  device={DEVICE}\ndata_root={DATA_ROOT}\nout_dir={OUT_DIR}")
if DEVICE == "cpu" and PRESET != "quick":
    print("WARNING: no GPU detected. The 'single'/'full' presets are only practical on a GPU.")


## 15.1 Fast data loading

In [ ]:
# ======================= 15.1 Fast data: load every tumour slice into RAM once =======================
import nibabel as nib
from torch.utils.data import Dataset, DataLoader, Subset

class SlicePool:
    """Loads all tumour-bearing axial slices once (FLAIR + whole-tumour mask, resized to 224x224).
    The stock loader re-reads whole NIfTI volumes for almost every shuffled slice, which is far too slow for 5 seeds."""
    def __init__(self, records, min_px=50, size=224):
        fl, mk, lab, pid, sl = [], [], [], [], []
        t0 = time.time()
        for k, rec in enumerate(records):
            flair = nib.load(rec.sequences["flair"]).get_fdata().astype(np.float32)
            seg = nib.load(rec.sequences["seg"]).get_fdata().astype(np.float32)
            keep = [s for s in range(seg.shape[2]) if (seg[:, :, s] > 0).sum() >= min_px]
            if not keep:
                continue
            f = torch.from_numpy(flair[:, :, keep]).permute(2, 0, 1).unsqueeze(1)
            m = torch.from_numpy((seg[:, :, keep] > 0).astype(np.float32)).permute(2, 0, 1).unsqueeze(1)
            fl.append(resize_batch(f, size).squeeze(1).numpy().astype(np.float16))
            mk.append(resize_mask(m, size).squeeze(1).numpy().astype(np.uint8))
            lab += [1 if rec.grade == "HGG" else (0 if rec.grade == "LGG" else -1)] * len(keep)
            pid += [rec.patient_id] * len(keep)
            sl += keep
            if (k + 1) % 25 == 0:
                print(f"  loaded {k+1}/{len(records)} patients ({time.time()-t0:.0f}s)")
        self.flair = np.concatenate(fl); self.mask = np.concatenate(mk)
        self.label = np.array(lab); self.pid = np.array(pid); self.sl = np.array(sl)
        self.by_pid = {}
        for i, p in enumerate(self.pid):
            self.by_pid.setdefault(p, []).append(i)
        self.by_pid = {p: np.array(v) for p, v in self.by_pid.items()}
        print(f"SlicePool: {len(self.label)} slices, {len(self.by_pid)} patients, "
              f"{self.flair.nbytes/1e9 + self.mask.nbytes/1e9:.2f} GB, {time.time()-t0:.0f}s")

class PoolView(Dataset):
    def __init__(self, pool, records):
        self.pool = pool
        ids = [pool.by_pid[r.patient_id] for r in records if r.patient_id in pool.by_pid]
        self.idx = np.concatenate(ids) if ids else np.array([], dtype=int)
        self.labels = pool.label[self.idx]; self.pids = pool.pid[self.idx]
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        fl = torch.from_numpy(self.pool.flair[j].astype(np.float32))
        x = torch.stack([torch.zeros_like(fl), fl])          # channel 0 = T1ce placeholder (unused), channel 1 = FLAIR
        m = torch.from_numpy(self.pool.mask[j].astype(np.float32))[None]
        return x, m, int(self.pool.label[j]), {"patient_id": str(self.pool.pid[j]), "slice_index": int(self.pool.sl[j])}

_POOL = {}
def _subsample_records(records, n, seed=123):
    if not n or n >= len(records):
        return records
    rng = np.random.default_rng(seed)
    h = [r for r in records if r.grade == "HGG"]; l = [r for r in records if r.grade != "HGG"]
    nh = max(1, int(round(n * 0.7))); nl = max(1, n - nh)
    h = [h[i] for i in rng.permutation(len(h))[:nh]]; l = [l[i] for i in rng.permutation(len(l))[:nl]]
    return h + l

def get_datasets(seed):
    """Returns train_ds, val_ds, test_ds, has_grades. NIfTI data with grades -> fast pooled datasets."""
    try:
        records = discover_patients_nifti(DATA_ROOT)
    except Exception:
        records = []
    if records and any(r.grade is not None for r in records):
        records = _subsample_records(records, cfg.max_patients)
        key = (DATA_ROOT, cfg.max_patients)
        if key not in _POOL:
            print(f"Loading {len(records)} patients into RAM (one-off) ...")
            _POOL[key] = SlicePool(records, cfg.min_tumour_pixels)
        tr, va, te = stratified_patient_split(records, seed=seed)
        pool = _POOL[key]
        ds = (PoolView(pool, tr), PoolView(pool, va), PoolView(pool, te))
        print(f"split seed {seed}: patients train/val/test = {len(tr)}/{len(va)}/{len(te)}; "
              f"slices = {len(ds[0])}/{len(ds[1])}/{len(ds[2])}")
        return (*ds, True)
    print("No grade-labelled NIfTI data found -> using the stock loader (segmentation only).")
    tr, va, te, has = build_datasets(DATA_ROOT, cfg.min_tumour_pixels, seed=seed)
    return tr, va, te, has
print("data helpers ready")


In [ ]:
# ======================= Patch: BraTS2020 patient 355 has a mis-named segmentation file =======================
import glob, os
if "_orig_discover" not in globals():
    _orig_discover = discover_patients_nifti
def discover_patients_nifti(root_dir):
    recs = _orig_discover(root_dir); out = []
    for r in recs:
        if not r.sequences.get("seg"):
            folder = os.path.dirname(r.sequences["flair"])
            c = glob.glob(os.path.join(folder, "*egm*.nii*")) + glob.glob(os.path.join(folder, "*seg*.nii*"))
            if c: r.sequences["seg"] = c[0]
        if r.sequences.get("seg"): out.append(r)
    print(f"patients with seg: {len(out)}/{len(recs)} | with grade: {sum(r.grade is not None for r in out)}")
    return out

## 15.2 Models (ablatable WFA-ResNet + classification baselines)

In [ ]:
# ======================= 15.2 Models: ablatable WFA-ResNet + classification baselines =======================
import torch.nn as nn, torch.nn.functional as F
import torchvision

class WFAResNetAblate(nn.Module):
    """Same architecture as WFAResNet (Section 3.5) with switches to drop the wavelet or spatial branch (ablation)."""
    def __init__(self, selected_feat_dim, use_wavelet=True, use_spatial=True, freq_dim=64, spatial_dim=64,
                 spatial_heads=8, num_dense_blocks=6, dense_growth=32, fusion_dim=1024, num_classes=2):
        super().__init__()
        self.use_wavelet, self.use_spatial = use_wavelet, use_spatial
        fin = selected_feat_dim
        if use_wavelet:
            self.wavelet_branch = WaveletAttentionBranch(freq_dim); fin += freq_dim
        if use_spatial:
            self.spatial_branch = SpatialMultiHeadAttentionBranch(1, spatial_dim, spatial_heads); fin += spatial_dim
        self.fusion = nn.Linear(fin, fusion_dim)
        self.head = DenseResidualHead(fusion_dim, num_dense_blocks, dense_growth, num_classes)
    def forward(self, I_seg, R):
        parts = []
        if self.use_spatial: parts.append(self.spatial_branch(I_seg))
        if self.use_wavelet: parts.append(self.wavelet_branch(I_seg)[0])
        parts.append(R)
        return self.head(F.relu(self.fusion(torch.cat(parts, 1))))

class SimpleCNN(nn.Module):
    def __init__(self, nc=2):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, nc))
    def forward(self, x): return self.fc(self.f(x))

class DNN(nn.Module):
    """Plain MLP on the full 4096-d VGG-16 feature vector (no feature selection, no attention)."""
    def __init__(self, d=4096, nc=2):
        super().__init__()
        self.net = nn.Sequential(nn.BatchNorm1d(d), nn.Linear(d, 512), nn.ReLU(True), nn.Dropout(0.3),
                                 nn.Linear(512, 128), nn.ReLU(True), nn.Linear(128, nc))
    def forward(self, v): return self.net(v)

def _one_channel(conv):
    new = nn.Conv2d(1, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=conv.bias is not None)
    with torch.no_grad():
        new.weight[:] = conv.weight.mean(1, keepdim=True)
    return new

def make_resnet(pretrained):
    try:
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    except Exception as e:
        print("  (ResNet-18 pretrained weights unavailable -> random init)", e); m = torchvision.models.resnet18(weights=None)
    m.conv1 = _one_channel(m.conv1); m.fc = nn.Linear(m.fc.in_features, 2); return m

def make_densenet(pretrained):
    try:
        m = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
    except Exception as e:
        print("  (DenseNet-121 pretrained weights unavailable -> random init)", e); m = torchvision.models.densenet121(weights=None)
    m.features.conv0 = _one_channel(m.features.conv0); m.classifier = nn.Linear(m.classifier.in_features, 2); return m

class ResDenseBlock(nn.Module):
    """Residual dense block: dense concatenation of conv layers + 1x1 fusion + identity shortcut."""
    def __init__(self, c, g=16, layers=3):
        super().__init__()
        self.layers = nn.ModuleList([nn.Sequential(nn.Conv2d(c + i * g, g, 3, padding=1), nn.BatchNorm2d(g), nn.ReLU(True))
                                     for i in range(layers)])
        self.fuse = nn.Conv2d(c + layers * g, c, 1)
    def forward(self, x):
        f = [x]
        for l in self.layers: f.append(l(torch.cat(f, 1)))
        return x + self.fuse(torch.cat(f, 1))

class ResDenseNet(nn.Module):
    def __init__(self, nc=2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 7, 2, 3), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2))
        self.b1, self.t1 = ResDenseBlock(32), nn.Sequential(nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True))
        self.b2, self.t2 = ResDenseBlock(64), nn.Sequential(nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True))
        self.b3 = ResDenseBlock(128)
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, nc))
    def forward(self, x):
        x = self.t1(self.b1(self.stem(x))); x = self.t2(self.b2(x)); return self.fc(self.b3(x))

print("models ready")


## 15.3 Helpers

In [ ]:
# ======================= 15.3 Helpers: banks, training, evaluation, refinement =======================
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def tolist(o):
    if isinstance(o, dict): return {k: tolist(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [tolist(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return o

# ---------- feature / image bank from a segmentation model ----------
@torch.no_grad()
def make_bank(seg_model, loader, vgg, device, max_samples=None):
    seg_model.eval(); vgg.eval()
    V, Y, I, P = [], [], [], []
    n = 0
    for x, mask, label, meta in loader:
        fpp = preprocess_slice(resize_batch(x[:, 1:2].to(device)))
        pm = (seg_model(fpp) >= 0.5).float()
        Iseg = fpp * pm
        v = vgg(resize_batch(Iseg, 224)).cpu().numpy()
        lab = label.numpy() if torch.is_tensor(label) else np.asarray(label)
        keep = np.where(lab != -1)[0]
        if len(keep) == 0: continue
        V.append(v[keep]); Y.append(lab[keep]); I.append(Iseg.cpu().half()[keep])
        P += [meta["patient_id"][k] for k in keep]
        n += len(keep)
        if max_samples and n >= max_samples: break
    V, Y, I = np.concatenate(V), np.concatenate(Y), torch.cat(I)
    if max_samples: V, Y, I, P = V[:max_samples], Y[:max_samples], I[:max_samples], P[:max_samples]
    return dict(v=V, y=Y, I=I, pid=np.array(P))

# ---------- generic classifier training / prediction ----------
def fit_classifier(model, Xtr, ytr, Xva, yva, fwd, epochs, lr, patience, bs, device, tag, wd=1e-5):
    model = model.to(device)
    ytr_t, yva_t = torch.as_tensor(ytr, dtype=torch.long), torch.as_tensor(yva, dtype=torch.long)
    cnt = np.bincount(ytr, minlength=2).astype(np.float32)
    w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    print(f"[{tag}] train n={len(ytr)} counts(benign,malignant)={cnt.astype(int).tolist()} "
          f"val counts={np.bincount(yva, minlength=2).tolist()} class weights={[round(float(a),3) for a in w.cpu()]}")
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=wd)
    best, best_state, bad, hist, t0 = float("inf"), None, 0, [], time.time()
    n = len(ytr)
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(n); tl = []
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            if len(idx) < 2: continue
            xb = [x[idx].to(device).float() for x in Xtr]
            opt.zero_grad()
            loss = F.cross_entropy(fwd(model, *xb), ytr_t[idx].to(device), weight=w)
            loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); num = den = 0.0
        with torch.no_grad():
            for i in range(0, len(yva), 128):
                xb = [x[i:i + 128].to(device).float() for x in Xva]; yb = yva_t[i:i + 128].to(device)
                li = F.cross_entropy(fwd(model, *xb), yb, reduction="none")
                num += (li * w[yb]).sum().item(); den += w[yb].sum().item()
        vl = num / max(den, 1e-8); hist.append((float(np.mean(tl)), vl))
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{tag}] epoch {ep}/{epochs} train={np.mean(tl):.4f} val={vl:.4f}")
        if vl < best: best, best_state, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: print(f"[{tag}] early stop at epoch {ep}"); break
    if best_state is not None: model.load_state_dict(best_state)
    _sync()
    return model, hist, time.time() - t0

@torch.no_grad()
def predict_probs(model, X, fwd, device, bs=128):
    model.eval(); out = []
    for i in range(0, len(X[0]), bs):
        xb = [x[i:i + bs].to(device).float() for x in X]
        out.append(torch.softmax(fwd(model, *xb), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def cls_report(y, p, pid):
    y, p = np.asarray(y), np.asarray(p); pred = (p >= 0.5).astype(int)
    rep = M.full_classification_report(y.tolist(), pred.tolist(), p.tolist())
    try: rep["AUC_exact"] = float(roc_auc_score(y, p))
    except ValueError: rep["AUC_exact"] = float("nan")
    rep["BalancedAccuracy"] = 0.5 * (rep["Recall_Sensitivity"] + rep["Specificity"])
    rep["test_malignant_fraction"] = float(y.mean()); rep["n_test_slices"] = int(len(y))
    # patient-level: mean malignant probability over a patient's slices
    ps = sorted(set(pid.tolist())); yp = np.array([y[pid == q][0] for q in ps]); pp = np.array([p[pid == q].mean() for q in ps])
    pr = (pp >= 0.5).astype(int)
    tp, tn, fp, fn = M.confusion_counts(yp, pr)
    rep["patient_level"] = dict(n_patients=len(ps), Accuracy=M.accuracy(tp, tn, fp, fn), Sensitivity=M.sensitivity_recall(tp, fn),
                                Specificity=M.specificity(tn, fp),
                                AUC_exact=float(roc_auc_score(yp, pp)) if len(set(yp)) == 2 else float("nan"))
    return rep

# ---------- segmentation evaluation (any predictor) ----------
@torch.no_grad()
def eval_seg(predict_fn, loader, device, example_idx=()):
    dsc, iou, hd, pids, ex, off, empty = [], [], [], [], {}, 0, []
    for x, mask, label, meta in loader:
        fpp = preprocess_slice(resize_batch(x[:, 1:2].to(device)))
        gt = resize_mask(mask.to(device))[:, 0].cpu().numpy()
        pm = predict_fn(fpp, off)
        for b in range(len(pm)):
            empty.append(float(pm[b].sum() == 0)); dsc.append(M.dice_score(pm[b], gt[b])); iou.append(M.iou_score(pm[b], gt[b]))
            hd.append(M.hausdorff_distance(pm[b], gt[b]) if pm[b].sum() > 0 and gt[b].sum() > 0 else float("nan"))
            if off + b in example_idx:
                ex[off + b] = (fpp[b, 0].cpu().numpy().astype(np.float16), gt[b].astype(np.uint8), pm[b].astype(np.uint8))
        pids += list(meta["patient_id"]) if "patient_id" in meta else [""] * len(pm)
        off += len(pm)
    dsc, iou, hd = np.array(dsc), np.array(iou), np.array(hd)
    summ = dict(DSC=float(dsc.mean()), IoU=float(iou.mean()), Jaccard=float(iou.mean()),
                HausdorffDistance=float(np.nanmean(hd)) if np.isfinite(hd).any() else float("nan"),
                n_slices=int(len(dsc)), n_hd=int(np.isfinite(hd).sum()), frac_empty_pred=float(np.mean(empty)))
    return summ, dict(dsc=dsc, iou=iou, hd=hd, pid=np.array(pids)), ex

def seg_predictor(model):
    model.eval()
    return lambda fpp, off: (model(fpp) >= 0.5).float()[:, 0].cpu().numpy()

# ---------- SC-GWO / GWO weight refinement of the segmentation head (Section 3.3) ----------
def refine_head(seg_model, val_ds, cfg, seed, device, pure_gwo=False):
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(val_ds, batch_size=cfg.batch_size_seg, shuffle=True, generator=g, num_workers=0)
    fl, mk, n = [], [], 0
    for x, m, _, _ in loader:
        fl.append(preprocess_slice(resize_batch(x[:, 1:2].to(device)))); mk.append(resize_mask(m.to(device))); n += len(x)
        if n >= cfg.refine_batch: break
    fl, mk = torch.cat(fl)[:cfg.refine_batch], torch.cat(mk)[:cfg.refine_batch]
    head = seg_model.head; w0, b0 = head.weight.detach().clone(), head.bias.detach().clone(); nw = w0.numel()
    @torch.no_grad()
    def apply(xv):
        t = torch.tensor(xv, dtype=w0.dtype, device=device)
        head.weight.copy_(w0 + 0.01 * t[:nw].view_as(w0)); head.bias.copy_(b0 + 0.01 * t[nw:].view_as(b0))
    @torch.no_grad()
    def fit(xv):
        apply(xv); return float(segmentation_loss(seg_model(fl), mk).item())
    seg_model.eval()
    base_fit = fit(np.zeros(nw + b0.numel()))
    kw = dict(p_min=0.0, p_max=0.0) if pure_gwo else {}
    opt = SCGWO(dim=nw + b0.numel(), fitness_fn=fit,
                config=SCGWOConfig(population_size=cfg.scgwo_weight_pop, iterations=cfg.scgwo_weight_iters,
                                   lower_bound=-1, upper_bound=1, seed=seed, **kw))
    bx, bf = opt.run()
    if bf < base_fit: apply(bx)            # keep the original weights if the search found nothing better
    else: apply(np.zeros(nw + b0.numel())); bf = base_fit
    print(f"[{'GWO' if pure_gwo else 'SC-GWO'} refinement] fitness {base_fit:.4f} -> {bf:.4f} (D={nw + b0.numel()}, batch={len(fl)})")
    return seg_model, dict(history=[float(h) for h in opt.history], start_fitness=base_fit, final_fitness=float(bf))
print("helpers ready")


## 15.4 Segmentation baselines (Faster R-CNN, YOLO)

In [ ]:
# ======================= 15.4 Segmentation baselines: Faster R-CNN, YOLO =======================
# Detectors predict boxes, so (as in the paper's detection-based baselines) a predicted box is filled in to give a mask.
# YOLO uses the instance-segmentation (-seg) variants from `ultralytics`, whose masks are used directly.
import importlib
def _try_import(name, pip_name=None):
    try: return importlib.import_module(name)
    except ImportError:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or name], check=True)
            return importlib.import_module(name)
        except Exception as e:
            print(f"  could not import/install {name}: {e}"); return None

def _boxes_from_masks(mk):
    boxes, keep = [], []
    for i, m in enumerate(mk):
        ys, xs = np.where(m > 0.5)
        if len(xs) == 0: continue
        boxes.append([xs.min(), ys.min(), max(xs.max() + 1, xs.min() + 2), max(ys.max() + 1, ys.min() + 2)]); keep.append(i)
    return torch.tensor(boxes, dtype=torch.float32), keep

def run_faster_rcnn(train_ds, test_loader, cfg, seed, device, example_idx):
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    seed_all(seed)
    try:
        wb = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if cfg.pretrained_baselines else None
        model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=wb, num_classes=2, min_size=224, max_size=224)
    except Exception as e:
        print("  (Faster R-CNN backbone weights unavailable -> random init)", e)
        model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=2, min_size=224, max_size=224)
    model = model.to(device)
    ds = train_ds
    if cfg.base_train_cap and len(ds) > cfg.base_train_cap:
        ds = Subset(ds, np.random.default_rng(seed).permutation(len(ds))[:cfg.base_train_cap].tolist())
    loader = DataLoader(ds, batch_size=cfg.base_batch_seg, shuffle=True, num_workers=0)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=cfg.base_lr_seg, momentum=0.9, weight_decay=5e-4)
    t0 = time.time()
    for ep in range(1, cfg.base_epochs_seg + 1):
        model.train(); ls = []
        for x, m, _, _ in loader:
            with torch.no_grad():
                fpp = preprocess_slice(resize_batch(x[:, 1:2].to(device))); gt = resize_mask(m.to(device))[:, 0].cpu().numpy()
            boxes, keep = _boxes_from_masks(gt)
            if not keep: continue
            imgs = [fpp[i].repeat(3, 1, 1) for i in keep]
            tg, k2 = [], 0
            for _ in keep:
                tg.append({"boxes": boxes[k2:k2 + 1].to(device), "labels": torch.ones(1, dtype=torch.int64, device=device)}); k2 += 1
            loss = sum(model(imgs, tg).values())
            opt.zero_grad(); loss.backward(); opt.step(); ls.append(loss.item())
        print(f"[Faster R-CNN] epoch {ep}/{cfg.base_epochs_seg} loss={np.mean(ls):.4f}")
    _sync(); train_time = time.time() - t0
    model.eval()
    @torch.no_grad()
    def pred(fpp, off):
        outs = model([f.repeat(3, 1, 1) for f in fpp]); res = np.zeros((len(fpp), 224, 224), np.uint8)
        for i, o in enumerate(outs):
            if len(o["boxes"]):
                x1, y1, x2, y2 = [int(round(v)) for v in o["boxes"][int(o["scores"].argmax())].tolist()]
                res[i, max(y1, 0):y2, max(x1, 0):x2] = 1
        return res
    return pred, train_time, model

def _export_yolo(ds, root, split, cap, seed, device):
    cv2 = _try_import("cv2", "opencv-python-headless")
    os.makedirs(f"{root}/images/{split}", exist_ok=True); os.makedirs(f"{root}/labels/{split}", exist_ok=True)
    idx = np.arange(len(ds))
    if cap and len(idx) > cap: idx = np.sort(np.random.default_rng(seed).permutation(len(ds))[:cap])
    sub = Subset(ds, idx.tolist()); n = 0
    for x, m, _, _ in DataLoader(sub, batch_size=32, shuffle=False, num_workers=0):
        with torch.no_grad():
            fpp = preprocess_slice(resize_batch(x[:, 1:2].to(device))).cpu().numpy()[:, 0]; gt = resize_mask(m)[:, 0].numpy()
        for b in range(len(fpp)):
            name = f"{n:06d}"
            cv2.imwrite(f"{root}/images/{split}/{name}.png", (np.clip(fpp[b], 0, 1) * 255).astype(np.uint8))
            cs, _ = cv2.findContours((gt[b] > 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            lines = []
            for c in cs:
                if len(c) >= 3 and cv2.contourArea(c) >= 4:
                    pts = (c.reshape(-1, 2) / 224.0).clip(0, 1).flatten()
                    lines.append("0 " + " ".join(f"{v:.5f}" for v in pts))
            open(f"{root}/labels/{split}/{name}.txt", "w").write("\n".join(lines))
            n += 1
    return n

def run_yolo(label, weights, train_ds, val_ds, test_ds, test_loader, cfg, seed, device, example_idx, work):
    ul = _try_import("ultralytics")
    if ul is None: raise RuntimeError("ultralytics not available")
    from ultralytics import YOLO
    cv2 = _try_import("cv2", "opencv-python-headless")
    root = os.path.join(work, "yolo_data"); 
    import shutil; shutil.rmtree(root, ignore_errors=True)
    cap = cfg.base_train_cap
    _export_yolo(train_ds, root, "train", cap, seed, device); _export_yolo(val_ds, root, "val", 2000, seed, device)
    n_te = _export_yolo(test_ds, root, "test", None, seed, device)
    open(os.path.join(root, "data.yaml"), "w").write(f"path: {os.path.abspath(root)}\ntrain: images/train\nval: images/val\nnames:\n  0: tumour\n")
    seed_all(seed)
    try: model = YOLO(weights if cfg.pretrained_baselines else weights.replace(".pt", ".yaml"))
    except Exception as e:
        print("  (YOLO weights unavailable -> from scratch)", e); model = YOLO(weights.replace(".pt", ".yaml"))
    t0 = time.time()
    model.train(data=os.path.join(root, "data.yaml"), epochs=cfg.base_epochs_seg, imgsz=224, batch=16, seed=seed,
                device=0 if device == "cuda" else "cpu", project=os.path.abspath(os.path.join(work, "yolo_runs")),
                name=label.replace(" ", "_"), exist_ok=True, verbose=False, plots=False, workers=2, patience=cfg.base_epochs_seg + 1)
    _sync(); train_time = time.time() - t0
    files = [f"{root}/images/test/{i:06d}.png" for i in range(n_te)]
    @torch.no_grad()
    def pred(fpp, off):
        rs = model.predict(source=files[off:off + len(fpp)], imgsz=224, conf=0.25, verbose=False, device=0 if device == "cuda" else "cpu")
        res = np.zeros((len(fpp), 224, 224), np.uint8)
        for i, r in enumerate(rs):
            if r.masks is not None and len(r.masks.data):
                mm = (r.masks.data.cpu().numpy() > 0.5).any(0).astype(np.uint8)
                res[i] = mm if mm.shape == (224, 224) else cv2.resize(mm, (224, 224), interpolation=cv2.INTER_NEAREST)
        return res
    return pred, train_time, model
print("segmentation baselines ready")


## 15.5 One complete seed

In [ ]:
# ======================= 15.5 Run one complete seed =======================
def _ms_per_slice(fn, reps=3):
    fn(); _sync(); t = time.time(); n = 0
    for _ in range(reps): n += fn()
    _sync(); return 1000.0 * (time.time() - t) / max(n, 1)

def run_seed(seed):
    path = os.path.join(OUT_DIR, f"seed{seed}.json")
    if os.path.exists(path) and not FORCE_RERUN:
        print(f"seed {seed}: already done -> skipping ({path})"); return
    print("\n" + "=" * 90 + f"\nSEED {seed}   preset={PRESET}\n" + "=" * 90)
    seed_all(seed); dev = DEVICE; T = {}; ERR = {}
    train_ds, val_ds, test_ds, has_grades = get_datasets(seed)
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size_seg, shuffle=True, generator=g, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size_seg, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size_seg, shuffle=False, num_workers=0)
    a = SimpleNamespace(**vars(args)); a.seed = seed; a.epochs_seg = cfg.epochs_seg; a.patience_seg = cfg.patience_seg
    a.lr_seg = cfg.lr_seg; a.batch_size_seg = cfg.batch_size_seg
    ex_idx = set(np.random.default_rng(1000 + seed).choice(len(test_ds), size=min(6, len(test_ds)), replace=False).tolist())
    R = dict(seed=seed, dataset=DATASET_NAME, preset=PRESET, config=tolist(vars(cfg)), has_grades=has_grades,
             n_train_slices=len(train_ds), n_val_slices=len(val_ds), n_test_slices=len(test_ds),
             segmentation={}, classification={}, ablation={}, baselines_cls={}, timing={}, histories={}, errors=ERR)
    ARR, EX = {}, {}
    def partial(): json.dump(tolist(R), open(path.replace(".json", ".partial.json"), "w"), indent=1)

    # ---------- Stage 1: Swin-ResSegNet gradient training ----------
    t = time.time(); seg_base = run_segmentation_training(train_loader, val_loader, a, dev); _sync()
    T["seg_train_s"] = time.time() - t
    # ---------- Stage 2: SC-GWO refinement ----------
    t = time.time(); seg_full, ref = refine_head(copy.deepcopy(seg_base), val_ds, cfg, seed, dev, pure_gwo=False)
    T["scgwo_refine_s"] = time.time() - t; R["histories"]["scgwo_refine"] = ref
    # ---------- segmentation metrics: proposed (+ no-refinement for the ablation) ----------
    s, arr, ex = eval_seg(seg_predictor(seg_full), test_loader, dev, ex_idx); R["segmentation"]["Proposed (Swin-ResSegNet + SC-GWO)"] = s
    ARR["seg__Proposed"] = arr; EX["Proposed"] = ex
    s0, arr0, _ = eval_seg(seg_predictor(seg_base), test_loader, dev); R["segmentation"]["Swin-ResSegNet without SC-GWO refinement"] = s0
    ARR["seg__NoRefine"] = arr0
    print(f"[proposed] DSC={s['DSC']:.4f} IoU={s['IoU']:.4f} HD={s['HausdorffDistance']:.2f} | no-refine DSC={s0['DSC']:.4f}")
    probe = torch.cat([preprocess_slice(resize_batch(x[:, 1:2].to(dev))) for x, *_ in [next(iter(test_loader))]])
    R["timing"]["infer_ms_per_slice_seg_proposed"] = _ms_per_slice(lambda: (seg_full(probe), len(probe))[1])
    partial()

    vgg = VGG16FeatureExtractor(pretrained=True, in_chans=1).to(dev).eval()
    full = None
    def cls_pipeline(seg_model, tag, gwo=False, use_wavelet=True, use_spatial=True, reuse=None):
        t_all = time.time()
        if reuse is None:
            # identical shuffled sample of slices for every variant (fixed generators) -> fair ablation comparisons
            tr_l = DataLoader(train_ds, batch_size=cfg.batch_size_seg, shuffle=True, generator=torch.Generator().manual_seed(seed + 2), num_workers=0)
            va_l = DataLoader(val_ds, batch_size=cfg.batch_size_seg, shuffle=True, generator=torch.Generator().manual_seed(seed + 1), num_workers=0)
            tr = make_bank(seg_model, tr_l, vgg, dev, cfg.max_feature_samples)
            va = make_bank(seg_model, va_l, vgg, dev, cfg.max_feature_samples)
            te = make_bank(seg_model, test_loader, vgg, dev, None)
            print(f"[{tag}] banks: train {np.bincount(tr['y'], minlength=2).tolist()} val {np.bincount(va['y'], minlength=2).tolist()} "
                  f"test {np.bincount(te['y'], minlength=2).tolist()} (benign, malignant)")
            kw = dict(p_min=0.0, p_max=0.0) if gwo else {}
            fcfg = SCGWOConfig(population_size=cfg.scgwo_fs_pop, iterations=cfg.scgwo_fs_iters, lower_bound=-4, upper_bound=4, seed=seed, **kw)
            tf = time.time(); mask_sel, n_sel, fs_fit, fs_hist = select_features(tr["v"], tr["y"], va["v"], va["y"], fcfg); t_fs = time.time() - tf
            sel = np.where(mask_sel > 0)[0]
            if len(sel) == 0: sel = np.arange(tr["v"].shape[1]); print(f"[{tag}] WARNING: empty feature subset -> using all features")
            print(f"[{tag}] {'GWO' if gwo else 'SC-GWO'} selected {len(sel)}/4096 features (fitness {fs_fit:.4f}, {t_fs/60:.1f} min)")
        else:
            tr, va, te, sel, n_sel, fs_fit, fs_hist, t_fs = (reuse[k] for k in ("tr", "va", "te", "sel", "n_sel", "fs_fit", "fs_hist", "t_fs"))
        Rtr, Rva, Rte = (torch.tensor(b["v"][:, sel]) for b in (tr, va, te))
        model = WFAResNetAblate(len(sel), use_wavelet, use_spatial)
        fwd = lambda m, I, R_: m(I, R_)
        model, hist, t_tr = fit_classifier(model, (tr["I"], Rtr), tr["y"], (va["I"], Rva), va["y"], fwd, cfg.epochs_cls,
                                           cfg.lr_cls, cfg.patience_cls, cfg.batch_size_cls, dev, tag)
        probs = predict_probs(model, (te["I"], Rte), fwd, dev)
        rep = cls_report(te["y"], probs, te["pid"]); rep["n_selected_features"] = int(len(sel))
        print(f"[{tag}] acc={rep['Accuracy']:.4f} spec={rep['Specificity']:.4f} AUC={rep['AUC_exact']:.4f} balAcc={rep['BalancedAccuracy']:.4f}")
        out = dict(rep=rep, probs=probs, y=te["y"], pid=te["pid"], hist=hist, fs_hist=[float(h) for h in fs_hist], t_fs=t_fs, t_tr=t_tr,
                   tr=tr, va=va, te=te, sel=sel, n_sel=n_sel, fs_fit=float(fs_fit), model=model)
        return out

    if has_grades:
        # ---------- Stage 3-4: classification (proposed) ----------
        full = cls_pipeline(seg_full, "WFA-ResNet (proposed)")
        R["classification"]["proposed"] = full["rep"]
        R["timing"].update(scgwo_feature_selection_s=full["t_fs"], wfa_train_s=full["t_tr"])
        R["histories"]["scgwo_fs"] = full["fs_hist"]; R["histories"]["wfa_train"] = full["hist"]
        ARR["cls_y"] = full["y"]; ARR["cls_pid"] = full["pid"]; ARR["cls_p__Proposed"] = full["probs"]
        _te, _R = full["te"], torch.tensor(full["te"]["v"][:, full["sel"]])
        R["timing"]["infer_ms_per_slice_cls_proposed"] = _ms_per_slice(
            lambda: (full["model"](_te["I"][:64].float().to(dev), _R[:64].float().to(dev)), 64)[1])
        partial()

        # ---------- classification baselines ----------
        if cfg.run_baselines:
            tr, va, te = full["tr"], full["va"], full["te"]
            specs = [("CNN", lambda: SimpleCNN(), "img", 1e-3), ("DNN", lambda: DNN(), "vec", 1e-3),
                     ("ResNet-18", lambda: make_resnet(cfg.pretrained_baselines), "img", 1e-4),
                     ("DenseNet-121", lambda: make_densenet(cfg.pretrained_baselines), "img", 1e-4),
                     ("ResDenseNet", lambda: ResDenseNet(), "img", 1e-3)]
            for name, mk, kind, lr in specs:
                try:
                    seed_all(seed); fwd = lambda m, x: m(x)
                    Xtr = (tr["I"],) if kind == "img" else (torch.tensor(tr["v"]),)
                    Xva = (va["I"],) if kind == "img" else (torch.tensor(va["v"]),)
                    Xte = (te["I"],) if kind == "img" else (torch.tensor(te["v"]),)
                    m_, h_, t_ = fit_classifier(mk(), Xtr, tr["y"], Xva, va["y"], fwd, cfg.base_epochs_cls, lr, cfg.base_patience,
                                                cfg.base_batch_cls, dev, name)
                    p_ = predict_probs(m_, Xte, fwd, dev); rep = cls_report(te["y"], p_, te["pid"])
                    n_par = sum(p.numel() for p in m_.parameters())
                    rep["n_params"] = int(n_par); R["baselines_cls"][name] = rep
                    R["timing"][f"train_s__{name}"] = t_
                    R["timing"][f"infer_ms_per_slice_cls__{name}"] = _ms_per_slice(lambda: (m_(Xte[0][:64].float().to(dev)), 64)[1])
                    ARR[f"cls_p__{name}"] = p_
                    print(f"[{name}] acc={rep['Accuracy']:.4f} spec={rep['Specificity']:.4f} AUC={rep['AUC_exact']:.4f}")
                    del m_
                except Exception as e:
                    import traceback; traceback.print_exc(); ERR[f"cls_baseline_{name}"] = repr(e)
                torch.cuda.empty_cache() if torch.cuda.is_available() else None
            partial()

        # ---------- ablation ----------
        if cfg.run_ablation and seed in cfg.ablation_seeds:
            abl = R["ablation"]; abl["Full model"] = dict(full["rep"], DSC=s["DSC"])
            for name, kw in [("w/o wavelet branch", dict(use_wavelet=False, reuse=full)),
                             ("w/o spatial attention branch", dict(use_spatial=False, reuse=full))]:
                try:
                    o = cls_pipeline(seg_full, "ablation: " + name, **kw); abl[name] = dict(o["rep"], DSC=s["DSC"]); ARR[f"cls_p__abl {name}"] = o["probs"]
                    del o
                except Exception as e:
                    import traceback; traceback.print_exc(); ERR["ablation_" + name] = repr(e)
            try:
                o = cls_pipeline(seg_base, "ablation: w/o SC-GWO refinement"); abl["w/o SC-GWO weight refinement"] = dict(o["rep"], DSC=s0["DSC"])
                ARR["cls_p__abl w/o SC-GWO weight refinement"] = o["probs"]; del o
            except Exception as e:
                import traceback; traceback.print_exc(); ERR["ablation_norefine"] = repr(e)
            try:
                seg_gwo, ref_g = refine_head(copy.deepcopy(seg_base), val_ds, cfg, seed, dev, pure_gwo=True)
                sg, arrg, _ = eval_seg(seg_predictor(seg_gwo), test_loader, dev); R["segmentation"]["Swin-ResSegNet + pure GWO refinement"] = sg
                o = cls_pipeline(seg_gwo, "ablation: pure GWO", gwo=True); abl["SC-GWO replaced by pure GWO"] = dict(o["rep"], DSC=sg["DSC"])
                ARR["cls_p__abl pure GWO"] = o["probs"]; R["histories"]["gwo_refine"] = ref_g; R["histories"]["gwo_fs"] = o["fs_hist"]; del o
            except Exception as e:
                import traceback; traceback.print_exc(); ERR["ablation_gwo"] = repr(e)
            partial()
    else:
        print("No grade labels -> classification, ablation and classification baselines skipped (segmentation only).")

    # ---------- segmentation baselines ----------
    if cfg.run_baselines:
        try:
            pr, tt, _m = run_faster_rcnn(train_ds, test_loader, cfg, seed, dev, ex_idx)
            sb, arrb, exb = eval_seg(pr, test_loader, dev, ex_idx); R["segmentation"]["Faster R-CNN (box as mask)"] = sb
            R["timing"]["train_s__Faster R-CNN"] = tt
            R["timing"]["infer_ms_per_slice_seg__Faster R-CNN"] = _ms_per_slice(lambda: (pr(probe, 0), len(probe))[1])
            ARR["seg__Faster R-CNN"] = arrb; EX["Faster R-CNN"] = exb
            print(f"[Faster R-CNN] DSC={sb['DSC']:.4f} IoU={sb['IoU']:.4f} HD={sb['HausdorffDistance']:.2f}"); del _m
        except Exception as e:
            import traceback; traceback.print_exc(); ERR["seg_faster_rcnn"] = repr(e)
        if cfg.run_yolo:
            for label, wts in cfg.yolo_models.items():
                try:
                    pr, tt, _m = run_yolo(label, wts, train_ds, val_ds, test_ds, test_loader, cfg, seed, dev, ex_idx, OUT_DIR)
                    sb, arrb, exb = eval_seg(pr, test_loader, dev, ex_idx); R["segmentation"][label] = sb
                    R["timing"][f"train_s__{label}"] = tt; ARR[f"seg__{label}"] = arrb; EX[label] = exb
                    R["timing"][f"infer_ms_per_slice_seg__{label}"] = _ms_per_slice(lambda: (pr(probe, 0), len(probe))[1])
                    print(f"[{label}] DSC={sb['DSC']:.4f} IoU={sb['IoU']:.4f} HD={sb['HausdorffDistance']:.2f}"); del _m
                except Exception as e:
                    import traceback; traceback.print_exc(); ERR[f"seg_{label}"] = repr(e)
        partial()

    R["timing"].update(T)
    R["timing"]["pipeline_total_s"] = float(sum(v for k, v in R["timing"].items() if k in
                       ("seg_train_s", "scgwo_refine_s", "scgwo_feature_selection_s", "wfa_train_s")))
    # ---------- save ----------
    npz = {}
    for k, v in ARR.items():
        if isinstance(v, dict):
            for kk, vv in v.items(): npz[f"{k}__{kk}"] = np.asarray(vv)
        else: npz[k] = np.asarray(v)
    for meth, ex in EX.items():
        for i, (f_, g_, p_) in ex.items():
            npz[f"ex__{meth}__{i}__pred"] = p_
            npz[f"ex__flair__{i}"] = f_; npz[f"ex__gt__{i}"] = g_
    np.savez_compressed(os.path.join(OUT_DIR, f"seed{seed}_arrays.npz"), **npz)
    json.dump(tolist(R), open(path, "w"), indent=1)
    pp = path.replace(".json", ".partial.json")
    if os.path.exists(pp): os.remove(pp)
    print(f"seed {seed} DONE -> {path}   errors: {list(ERR) or 'none'}")
print("run_seed ready")


In [ ]:
# ======================= RUN: YOLO v9 + YOLO v8 on BOTH datasets (long-running) =======================
import subprocess, sys, json, zipfile, os, shutil, gc
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"]); import ultralytics

cfg.run_yolo = True
seed = 0                                   # same patient-level split / test slices as the main runs
rng = np.random.default_rng(0)
def dsc_ci(arr, n=500):                    # 95% patient-level bootstrap interval (same as the main tables)
    pid, d = arr["pid"], arr["dsc"]; groups = [d[pid == p] for p in np.unique(pid)]
    m = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n)]
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

ALL = {}
for name in ["BraTS2020", "BraTS2019"]:
    if name not in ROOTS:
        print(f"!! {name} not found under /kaggle/input -> skipped (Add Input for it)"); continue
    DATASET_NAME, DATA_ROOT = name, ROOTS[name]
    OUT_DIR = os.path.join(OUT_ROOT, name, "yolo"); os.makedirs(OUT_DIR, exist_ok=True)
    print("\n" + "=" * 90 + f"\n{name}   root={DATA_ROOT}\n" + "=" * 90)
    seed_all(seed); dev = DEVICE
    train_ds, val_ds, test_ds, has_grades = get_datasets(seed)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size_seg, shuffle=False, num_workers=0)
    ex_idx = set(np.random.default_rng(1000 + seed).choice(len(test_ds), size=min(6, len(test_ds)), replace=False).tolist())
    probe = torch.cat([preprocess_slice(resize_batch(x[:, 1:2].to(dev))) for x, *_ in [next(iter(test_loader))]])
    out, npz = {}, {}
    for label, wts in cfg.yolo_models.items():          # YOLO v9 (yolov9c-seg) and YOLO v8 (yolov8m-seg)
        try:
            pr, tt, _m = run_yolo(label, wts, train_ds, val_ds, test_ds, test_loader, cfg, seed, dev, ex_idx, OUT_DIR)
            sb, arrb, exb = eval_seg(pr, test_loader, dev, ex_idx)
            ms = _ms_per_slice(lambda: (pr(probe, 0), len(probe))[1])
            out[label] = dict(metrics=sb, dsc_ci95=dsc_ci(arrb), train_min=tt / 60, infer_ms_per_slice=ms)
            for k, v in arrb.items(): npz[f"{label}__{k}"] = np.asarray(v)
            print(f"[{name} | {label}] DSC={sb['DSC']:.4f} IoU={sb['IoU']:.4f} HD={sb['HausdorffDistance']:.2f} "
                  f"empty={100*sb['frac_empty_pred']:.1f}% | DSC CI={out[label]['dsc_ci95']} | train {tt/60:.1f} min | {ms:.2f} ms/slice")
            del _m
        except Exception as e:
            import traceback; traceback.print_exc(); out[label] = dict(error=repr(e))
    out["_info"] = dict(dataset=name, n_train=len(train_ds), n_test=len(test_ds), epochs=cfg.base_epochs_seg,
                        train_cap=cfg.base_train_cap, pretrained=cfg.pretrained_baselines)
    ALL[name] = out
    json.dump(tolist(out), open(f"./YOLO_{name}.json", "w"), indent=1); np.savez(f"./YOLO_{name}_arrays.npz", **npz)
    for d_ in ("yolo_data", "yolo_runs"): shutil.rmtree(os.path.join(OUT_DIR, d_), ignore_errors=True)   # keep the output small
    _POOL.clear(); gc.collect(); torch.cuda.empty_cache()

zp = "./YOLO_RESULTS_both.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("./YOLO_*.json") + glob.glob("./YOLO_*_arrays.npz"): z.write(f)
print("\n===== COPY EVERYTHING BELOW AND SEND IT TO ME =====")
print(json.dumps(tolist(ALL), indent=1))
print("zip:", zp, os.path.getsize(zp), "bytes")